In [ ]:
################################## Test the bbknn result based on endothelial lineage
import pickle 
import os
import re
import pandas as pd
import numpy as np
import sys
import scanpy as sc
# import scatlasvae  ## 
import scipy.sparse as sp
import scipy
import bbknn
import harmonypy as hm
import gc
import weakref
import resource

import sys, os
print("=== 当前 Jupyter 使用的信息 ===")
print("Python 路径：", sys.executable)
print("Python 版本：", sys.version.split()[0])
print("Conda 环境：", os.environ.get('CONDA_DEFAULT_ENV'))
import socket
print(socket.gethostname())
!hostname
import os, platform, socket
print("Node 主机名:", socket.gethostname())
print("系统版本:", platform.platform())
print("CPU 核心数:", os.cpu_count())
print("当前工作目录:", os.getcwd())
os.chdir('/data/project/chenfeiwang')


In [ ]:
cell_lineage_dict = {
    "T": [
        "CD3D", "CD3E", "CD3G",  # T细胞核心标记
        "CD4", "CD8A", "CD8B",    # CD4+/CD8+ T亚群
        "FOXP3", "IL2RA",          # Treg细胞标记
        "TRAC", "TRBC1", "TRBC2"  # TCR基因
    ],   # Endo中有 ['CD3D', 'CD3E', 'CD4', 'TRAC', 'TRBC2']
    "B": [
        "CD19", "CD79A", "CD79B",  # B细胞核心标记
        "MS4A1",                   # CD20
        "IGHD", "IGHM",            # B细胞受体
        "CD27", "CD38"             # 记忆B/浆细胞标记
    ],   # Endo中有 ['CD79B', 'CD38']
    "Fibro": [
        "COL1A1", "COL1A2",        # 胶原蛋白
        "COL3A1", "COL6A1", 
        "DCN", "LUM",              # 基质蛋白
        "FAP", "PDPN",             # 成纤维活化标记
        "ACTA2"                    # 肌成纤维标记（α-SMA）
    ],    # Endo中有 ['COL1A1', 'COL1A2', 'COL6A1', 'DCN', 'PDPN', 'ACTA2']
    "Endo": [
        "PECAM1", "CDH5",         # 内皮粘附分子
        "VWF", "CLDN5",           # Weibel-Palade小体标记
        "SELE", "ICAM1",           # 活化内皮标记
        "FLT1", "KDR"              # VEGF受体
    ],     # Endo中有 ['PECAM1', 'ICAM1', 'FLT1']
    "MoMac": [
        "CD14", "CD68",            # 单核/巨噬核心标记
        "FCGR3A", "CD16",          # 非经典单核
        "LYZ", "CST3",             # 溶酶体相关
        "S100A8", "S100A9",       # 钙结合蛋白
        "ITGAM"                    # CD11b
    ],     # Endo中有 ['CD14', 'CD68', 'LYZ', 'CST3', 'S100A8', 'S100A9', 'ITGAM']
    "Epi": [
        "EPCAM", "CDH1",           # 上皮粘附分子
        "KRT8", "KRT18",           # 角蛋白
        "MUC1", "TFF3",            # 分泌蛋白
        "SFN", "CLDN3"             # 紧密连接蛋白
    ]    # Endo中有 ['EPCAM', 'CDH1', 'KRT8', 'KRT18', 'MUC1']
}


In [ ]:
#### Endo中用于注释的 subtype marker list
Endo_marker_dict_full = {
    "SMC1": ["ACTA2", "MYH11", "MYL9", "PLN", "RERGL", "SOD3", "TAGLN", "TPM2"],    #  Endo中有 ['MYH11', 'TAGLN', 'ACTA2', 'TPM2', 'MYL9', 'SOD3']
    "Peri1": ["BGN", "COL1A1", "COL1A2", "COL3A1", "COL6A1", "COL6A3", "RGS5", "THY1"],       # Endo中有 ['COL1A1', 'COL1A2', 'RGS5', 'COL6A1', 'THY1', 'COL6A3', 'BGN']
    "LEC1": ["CCL21", "EFEMP1", "LYVE1", "MMRN1", "PROX1", "TFF3",'FLT4','RELN'],   #  Endo中有 ['EFEMP1']
    "VenEC1": ["ACKR1", "CCL14", "CLU", "SELE"],   #  Endo中有 ['CLU']
    "CapEC1": ["COL15A1", "FLT1", "HSPG2", "INSR", "PCDH17", "PLVAP", "RGCC", "VWA1"],   #  Endo中有 ['FLT1', 'HSPG2', 'RGCC', 'INSR', 'VWA1', 'PCDH17']
    "ArtEC1": ["ARL15", "CXCL12", "FBLN5", "IGFBP3", "SRGN"]   #  Endo中有 ['CXCL12', 'IGFBP3', 'SRGN', 'FBLN5', 'ARL15']
    # "EndoMT-I": [ "BGN", "NOTCH3", "NDUFA4L2"],
    # "EndoMT-II": ['APOD','CFD', 'DCN', 'CXCL14']
}

Endo_marker_dict_full_new = {
    "SMC_1": ["ACTA2", "MYH11", "MYL9", "SOD3", "TAGLN", "TPM2"],
    "Peri_1": ["COL6A1", "COL6A3", "PDGFRB", "RGS5", "THY1"],
    "LEC_1": ["EFEMP1", 'PDPN'],
    "VenEC_1": [ "CLU", "NRP2", "VCAM1"],
    "CapEC_1": ["EGFL7", "FLT1", "HSPG2", 'INSR', 'PCDH17', 'VWA1' ],
    "ArtEC_1": ["CXCL12", "FBLN5", 'IGFBP3', 'SRGN','ARL15', 'CXCR4', 'HEY1'],
    "EndoMT-I_1": ["CAMK2N1", "CD36", "NOTCH3",'BGN', 'NDUFA4L2','SNAI1', 'VIM', 'COL5A1'],
    "EndoMT-II_1": ['MMP2','CFD', "DCN", "FN1"]
}

#### Endo 中用R画每个subtype的marker dotplot的 marker list
Endo_marker_dict = {
    "SMC": ["TAGLN"],  
    "Peri": ["COL1A1", "COL1A2", "COL3A1", "RGS5"],
    "LEC": ["CCL21", "MMRN1", "TFF3"],
    "VenEC": ["ACKR1", "CCL14", "CLU"],
    "CapEC": ["FLT1", "HSPG2", "PLVAP"],
    "ArtEC": ["CXCL12", "IGFBP3", "SRGN"],
    "EndoMT-I": ["BGN", "NDUFA4L2", "NOTCH3"],   #  Endo中有 ['BGN', 'NOTCH3', 'NDUFA4L2']
    "EndoMT-II": ["APOD", "CFD", "CXCL14", "DCN"]   #    Endo中有 ['CFD', 'DCN']
} ## 按照DeepSeek的回答，最准确的一个是将 RGS5 归到pericytes 中,虽然EndoMT-I在转化过程中可能会表达 RGS5，另外 血管平滑肌细胞( vSMC) 也会表达「特别是大血管中」
#### 主要属于 Peri (周细胞)，其次是 SMC (平滑肌细胞) 和 EndoMT-I（若发生转化）


####### Endo_marker_dict_optimized, 炫鑫给我最新的
Endo_marker_dict_optimized = {
    "SMC_o": ["ACTA2", "MYH11", "MYL9", "PLN", "TPM2"], #TAGLN, RERGL, SOD3可以加上     {  Endo中有 ['MYH11', 'ACTA2', 'TPM2', 'MYL9'] }             
    "Peri_o": ["COL6A1", "COL6A3", "PDGFRB", "RGS5", "THY1"], #添加了PDGFRB，去掉了COL1A1, COL3A1, COL1A2可能与肿瘤相关          {  Endo中有 ['COL6A1', 'THY1', 'COL6A3', 'PDGFRB', 'RGS5'] }      
    "LEC_o": ["EFEMP1", "LYVE1", "MMRN1", "PROX1",'FLT4','RELN'], #去掉了CCL21, TFF3可能与肿瘤相关           {  Endo中有 ['EFEMP1'] }      
    "VenEC_o": ["ACKR1", "CLU"], #添加了AQP1（静脉内皮标记基因），去掉了CCL14, SELE，是血管黏附与癌症相关的基因;AQP1 并不是.               {  Endo中有 ['CLU'] }      
    "CapEC_o": ["FLT1", "HSPG2", "PLVAP", "RGCC"], #去掉了INSR, VWA1, PCDH17, COL15A1（肿瘤血管生成、基质重构相关）           {  Endo中有 ['RGCC', 'FLT1', 'HSPG2'] }      
    "ArtEC_o": ["CXCL12", "FBLN5", "GJA5", "SEMA3G"], #添加了SEMA3G, GJA5，去掉了IGFBP3, SRGN, ARL15（肿瘤血管调控、癌微环境相关）                {  Endo中有 ['CXCL12', 'FBLN5'] }      
    "EndoMT-I_o": ["CAMK2N1", "CD36", "NOTCH3"], #添加了CAMK2N1，CD36，但这俩在印老师paper中，也在peri中表达，只不过特异性不如EndoMT；               {  Endo中有 ['NOTCH3', 'CAMK2N1', 'CD36'] }      
    "EndoMT-II_o": ["CXCL14", "DCN"] #添加LUM，去除APOD, CFD（肿瘤脂代谢、基质重塑相关）                          {  Endo中有 ['DCN'] }      
}




# 权威重构版 Endothelial 8亚型 Marker Dictionary（顶刊公认、无争议、特异性强、适配泛数据集）,在上面 optimized基础上问 GPT，给出修正和更改 
Endo_marker_dict_authoritative = {
  "SMC_o": ["ACTA2", "MYH11", "MYL9", "SOD3", "TAGLN", "TPM2"],     # 血管平滑肌细胞 SMC：经典谱系特异，无跨界高表达                   {  Endo中有 ['MYH11', 'ACTA2', 'TPM2', 'MYL9', 'TAGLN', 'SOD3'] }  
  "Peri_o": ["CSPG4", "NG2", "PDGFRB", "RGS5", "THY1"],    # 周细胞 Pericyte：专一性强，避开成纤维/EndoMT共表达基因                    {  Endo中有 ['PDGFRB', 'RGS5', 'THY1'] }  
  "LEC_o": ["EFEMP1", "LYVE1", "MMRN1", "PDPN", "PROX1",'FLT4','RELN'],    # 淋巴管内皮 LEC：金标准组合，稳态+疾病通用                               {  Endo中有 ['EFEMP1', 'PDPN'] }  
  "VenEC_o": ["ACKR1", "NRP2", "SELE", "VCAM1"],  # 静脉内皮 VenEC：扩充数量、全部谱系特异，剔除广谱应激基因                           {  Endo中有 ['VCAM1', 'NRP2'] }  
  "CapEC_o": ["EGFL7", "FLT1", "HSPG2", "KDR", "PLVAP"],  # 毛细血管内皮 CapEC：微血管专属，稳态/肿瘤毛细血管均适用                    {  Endo中有 ['FLT1', 'HSPG2', 'EGFL7'] }  
  "ArtEC_o": ["CXCR4", "FBLN5", "GJA5", "HEY1", "SEMA3G"],   # 动脉内皮 ArtEC：动脉特异性通路+结构Marker，无免疫/基质杂表达            {  Endo中有 ['FBLN5', 'CXCR4', 'HEY1'] }  
  "EndoMT-I_o": ["COL5A1", "SNAI1", "TWIST1", "VIM"],     # EndoMT-I 内皮间质转化I型：避开Peri/SMC重叠基因，EndoMT专属                 {  Endo中有 ['SNAI1', 'VIM', 'COL5A1'] }  
  "EndoMT-II_o": ["CXCL14", "FN1", "LUM", "MMP2"]     # EndoMT-II 内皮间质转化II型：特异性强，与成纤维/Peri可区分                      {  Endo中有 ['FN1', 'MMP2'] }  
}  



############ functional molecular signature dict
score_dict = {
    "AP": ["CD74", "CD83", "H2-Q4", "HLA-DPA1", "HLA-DQA", "HLA-DQA1", "HLA-DRA", "HLA-DRB1"],     #    {  Endo中有 ['CD74', 'HLA-DQA1', 'CD83', 'HLA-DRA', 'HLA-DPA1', 'HLA-DRB1'] }  
    "Tip": ["ADM", "ANGPT2", "ANGPTL2", "APLN", "CXCR4", "ESM1", "FSCN1", "INSR", "LXN", "PDGFB", "PGF", "PLXND1", "PXDN"],      #   {  Endo中有 ['ANGPT2', 'FSCN1', 'PGF', 'PLXND1', 'ADM', 'PDGFB', 'CXCR4', 'PXDN', 'LXN', 'ANGPTL2', 'INSR'] }  
    "Stalk": ["ACKR1", "ADIRF", "CCL2", "CLU", "HLA-DRA", "IL33", "POSTN", "SELE"]       #  {  Endo中有 ['CLU', 'ADIRF', 'CCL2', 'HLA-DRA'] }  
}


In [ ]:
#### 最终 short & long 用下面这个, 里面的gene
Endo_marker_dict_full_latest = {
  "SMC_1": ["ACTA2", "MYH11", "MYL9", "SOD3", "TAGLN", "TPM2"],
  "Peri_1": ["COL6A1", "COL6A3", "PDGFRB", "RGS5", "THY1"],
  "LEC_1": ["EFEMP1", 'PDPN'],
  "VenEC_1": [ "CLU", "NRP2", "VCAM1"],
  "CapEC_1": ["EGFL7", "FLT1", "HSPG2", 'INSR', 'PCDH17','PTPRM', 'VWA1','TM4SF1' ],
  "ArtEC_1":["CXCL12",'PTPRM','EFNB2', "FBLN5", 'IGFBP3', 'SRGN','ARL15', 'CXCR4', 'HEY1']  , # 
  "EndoMT-I_1": ["CAMK2N1", "CD36", "NOTCH3",'BGN', 'NDUFA4L2','SNAI1', 'VIM', 'COL5A1'],
  "EndoMT-II_1": ['MMP2','CFD', "DCN", "FN1"]
}

Endo_marker_dict_latest = {
  "SMC": [ "MYL9",  "TAGLN", "TPM2"],
  "Peri": ["COL6A1", "COL6A3",  "RGS5"],
  "LEC": ["EFEMP1", 'PDPN'],  
  "VenEC": [ "CLU", "NRP2"],
  "CapEC": ["EGFL7", "FLT1",'PTPRM', "HSPG2", 'INSR','TM4SF1' ],
  "ArtEC":["CXCL12", 'PTPRM', 'SRGN','ARL15'] , #  'PTPRM'在ArtEC中最高最特异表达, 在CapEC中最稳定表达, 但要结合使用:PTPRM、EFNB2
  "EndoMT-I": ['VIM', "CAMK2N1", "CD36",  'COL5A1'],
  "EndoMT-II": ['MMP2','CFD', "DCN", "FN1"] 
}

   

In [ ]:
#### 最终 short & long 用下面这个, 6 个 subtype
Endo_marker_6dict_full = {
  "MCs_1": ['PDGFRB',"ACTA2", "MYH11", "MYL9", "SOD3", "TAGLN", "TPM2"],
 # "Peri_1": ["COL6A1", "COL6A3", "PDGFRB", "RGS5", "THY1"],
  "LEC_1": ["EFEMP1", 'PDPN'],  # []
  "VenEC_1": [ "CLU", "NRP2",'NR2F2', "VCAM1"],
  "CapEC_1": ["EGFL7", "FLT1", "HSPG2", 'INSR', 'PCDH17','PTPRM', 'VWA1','TM4SF1','RGCC' ],
  "ArtEC_1":["CXCL12",'PTPRM','EFNB2', "FBLN5", 'IGFBP3', 'SRGN','ARL15', 'CXCR4', 'HEY1']  , # 
  "EndoMT_1": ["CAMK2N1", "CD36", "NOTCH3",'BGN', 'NDUFA4L2','SNAI1', 'VIM', 'COL5A1'],
 #  "EndoMT": ['MMP2','CFD', "DCN", "FN1"]
}

Endo_marker_6dict = {
  "MCs": [ 'PDGFRB','RGS5' ,"MYL9","TAGLN","TPM2" ],
  "LEC": ["EFEMP1", 'PDPN'],
  "VenEC": [ "CLU", "NRP2",'NR2F2'],
  "CapEC": ["EGFL7", "FLT1",'PTPRM', "HSPG2", 'INSR','TM4SF1','RGCC'],
  "ArtEC":["CXCL12", 'PTPRM', 'SRGN','ARL15'] , #  'PTPRM'在ArtEC中最高最特异表达, 在CapEC中最稳定表达, 但要结合使用:PTPRM、EFNB2
  "EndoMT": ['VIM', "CAMK2N1", "CD36",  'COL5A1'],
  # "EndoMT": ['MMP2','CFD', "DCN", "FN1"] 
}

   


In [ ]:
## 重新再算一下
Fib_marker_deg = ['DCN', 'COL1A2', 'LUM', 'MGP', 'COL1A1', 'COL3A1', 'CALD1', 'IGFBP7', 'COL6A2', 'SPARC']  
Endo_marker_deg =['VWF', 'PECAM1', 'TM4SF1', 'RAMP2', 'GNG11', 'CDH5', 'TEK', 'KDR']    
stromal_marker = {
    'Fib_deg': Fib_marker_deg,    # Endo 中有 ['DCN', 'COL1A2', 'MGP', 'COL1A1', 'CALD1', 'IGFBP7', 'COL6A2', 'SPARC']
    'Endo_deg': Endo_marker_deg     # Endo 中有 ['PECAM1', 'TM4SF1', 'RAMP2', 'GNG11']
}
gene_marker_list = [gene for gene in Fib_marker_deg if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Fib_deg')
gene_marker_list = [gene for gene in Endo_marker_deg if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Endo_deg')



In [ ]:
import scanpy as sc
import numpy as np

def calculate_simple_scores(adata, marker_dict, score_prefix=''):
    """
    计算简单的平均表达得分（绝对非负）
    """
    for key, genes in marker_dict.items():
        score_name = f"{score_prefix}{key}" if score_prefix else key
        
        # 过滤存在的基因
        valid_genes = [g for g in genes if g in adata.var_names]
        
        if len(valid_genes) == 0:
            print(f"Warning: No genes found for {key}")
            adata.obs[score_name] = 0
            continue
        
        print(f"Calculating {score_name} with {len(valid_genes)} genes")
        print(sorted(valid_genes))
        # 计算平均表达
        # 注意：对于log-normalized数据，可以直接平均
        expr_data = adata[:, valid_genes].X
        if scipy.sparse.issparse(expr_data):
            scores = expr_data.mean(axis=1).A1  # A1 converts to 1D array
        else:
            scores = expr_data.mean(axis=1)
        
        # 确保非负
        scores = np.maximum(scores, 0)
        adata.obs[score_name] = scores
        print(f"  Range: [{scores.min():.4f}, {scores.max():.4f}]")

# 计算所有得分
print("Calculating all fibroblast subtype scores...")
cols_to_remove = ['SMC_s', 'Peri_s', 'VenEC_s', 'CapEC_s', 'ArtEC_s','EndoMT-I_s', 'EndoMT-II_s']
adata.obs.drop(columns=cols_to_remove, inplace=True) 
calculate_simple_scores(adata,cell_lineage_dict ) 
calculate_simple_scores(adata,stromal_marker )  ## 
calculate_simple_scores(adata, Endo_marker_dict)
calculate_simple_scores(adata, Endo_marker_dict_full)
calculate_simple_scores(adata, Endo_marker_dict_full_new) 
calculate_simple_scores(adata, Endo_marker_dict_full_latest) # 
calculate_simple_scores(adata, Endo_marker_dict_latest) 
calculate_simple_scores(adata,Endo_marker_dict_optimized )
calculate_simple_scores(adata,Endo_marker_dict_authoritative )  
calculate_simple_scores(adata,score_dict ) #  
calculate_simple_scores(adata, Endo_marker_6dict)
calculate_simple_scores(adata, Endo_marker_6dict_full)




In [ ]:
Endo_markers = ['ACKR1', 'ACTA2', 'APLN', 'APOD', 'ARL15', 'BGN', 'BMX', 'CCL14', 'CCL21', 'CD105', 'CD144', 'CD54', 'CD62E', 'CDH5', 'CFD', 'CHGB', 'CLU', 'COL15A1', 'COL1A1', 'COL1A2', 'COL3A1', 'COL6A1', 'COL6A3', 'CXCL12', 
    'CXCL14', 'CXCR4', 'DCN', 'EFEMP1', 'ESM1', 'FBLN5', 'FLK1', 'FLT1', 'GNG11', 'HSPG2', 'IGFBP3', 'IGKC', 'INSR', 'KDR', 'LYVE1', 'MMRN1', 'MYH11', 'MYL9', 'NDUFA4L2', 'NOTCH3', 'NR2F2', 'PCDH17', 'PDPN', 'PECAM1', 'PLN', 
'PLVAP', 'PROX1', 'RAMP2', 'RERGL', 'RGCC', 'RGS5', 'SELE', 'SFN', 'SOD3', 'SPARC', 'SRGN', 'TAGLN', 'TCIM', 'TEK', 'TFF3', 'THY1', 'TM4SF1', 'TPM2', 'VCAM1', 'VWA1', 'VWF','EGFL7', 'PTPRM', 'TRIO','ASAP1','PTK2','UTRN','PECAM1','VWF','EFNB2','CDH5']    # 加上 EGFL7, PTPRM, TRIO、ASAP1、PTK2、UTRN、PECAM1、VWF、CDH5
genes_union = set().union(*Endo_marker_dict.values(), *Endo_marker_dict_full.values(),*Endo_marker_dict_optimized.values(), *score_dict.values(), *Endo_marker_dict_authoritative.values() )
Endo_marker_all = list(set(Endo_marker_deg) | set(Endo_markers) | set(genes_union))




In [ ]:
gene_marker_list = [gene for gene in Endo_marker_dict['SMC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='SMC')
gene_marker_list = [gene for gene in Endo_marker_dict['Peri'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Peri')
gene_marker_list = [gene for gene in Endo_marker_dict['LymEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='LEC')
gene_marker_list = [gene for gene in Endo_marker_dict['VenEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='VenEC')
gene_marker_list = [gene for gene in Endo_marker_dict['CapEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='CapEC')
gene_marker_list = [gene for gene in Endo_marker_dict['ArtEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='ArtEC')   
gene_marker_list = [gene for gene in Endo_marker_dict['EndoMT-I'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-I')
gene_marker_list = [gene for gene in Endo_marker_dict['EndoMT-II'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-II') 


In [ ]:
gene_marker_list = [gene for gene in cell_marker_dict['SMC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='SMC_s')
gene_marker_list = [gene for gene in cell_marker_dict['PC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Peri_s')
gene_marker_list = [gene for gene in cell_marker_dict['LEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='LEC_s')
gene_marker_list = [gene for gene in cell_marker_dict['VenEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='VenEC_s')
gene_marker_list = [gene for gene in cell_marker_dict['CapEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='CapEC_s')
gene_marker_list = [gene for gene in cell_marker_dict['ArtEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='ArtEC_s')   
gene_marker_list = [gene for gene in cell_marker_dict['EndoMT-I'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-I_s')
gene_marker_list = [gene for gene in cell_marker_dict['EndoMT-II'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-II_s') 


In [ ]:
gene_marker_list = [gene for gene in cell_marker_dict_optimized['SMC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='SMC_1')
gene_marker_list = [gene for gene in cell_marker_dict_optimized['Peri'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Peri_1')
gene_marker_list = [gene for gene in cell_marker_dict_optimized['LymEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='LEC_1')
gene_marker_list = [gene for gene in cell_marker_dict_optimized['VenEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='VenEC_1')
gene_marker_list = [gene for gene in cell_marker_dict_optimized['CapEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='CapEC_1')
gene_marker_list = [gene for gene in cell_marker_dict_optimized['ArtEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='ArtEC_1')   
gene_marker_list = [gene for gene in cell_marker_dict_optimized['EndoMT-I'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-I_1')
gene_marker_list = [gene for gene in cell_marker_dict_optimized['EndoMT-II'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-II_1') 


In [ ]:
######## 为了更好的看 Endo_marker_all 中的基因在每个 Leiden中的 top 表达基因，因为看单个基因的UMAP肉眼容易出错，下面进行定量查看,另外加上 leiden, Celltype, 以及对基因进行 group 等操作
import numpy as np
import pandas as pd
from scipy.sparse import issparse

### Endo_marker_all & Endo_marker_dict 上面都有, 另外 Endo_marker_dict_full 中必有 Endo_marker_dict 的基因

# 1. 创建基因分类映射
gene_to_subtype = {}
for subtype, genes in Endo_marker_6dict_full.items():
    for gene in genes:
        if gene in Endo_marker_all:
            gene_to_subtype[gene] = subtype

# 为不在任何subtype中的基因标记为'Other'
for gene in Endo_marker_all:
    if gene not in gene_to_subtype:
        gene_to_subtype[gene] = 'Other'

# 2. 定义subtype的顺序
subtype_order = ['MCs_1', 'LEC_1', 'VenEC_1', 'CapEC_1', 'ArtEC_1', 'EndoMT_1','Other']      # subtype 先不加

# 3. 对基因进行排序
genes_by_subtype = {subtype: [] for subtype in subtype_order}
for gene in Endo_marker_all:
    subtype = gene_to_subtype[gene]
    genes_by_subtype[subtype].append(gene)

# 按指定顺序拼接基因列表
sorted_genes = []
for subtype in subtype_order:
    sorted_genes.extend(genes_by_subtype[subtype])

print(f"Sorted {len(sorted_genes)} genes by subtype")

# 4. 从adata中高效提取表达数据
print("Extracting expression data...")

# 确保所有基因都在adata中
existing_genes = [gene for gene in sorted_genes if gene in adata.var_names]
missing_genes = set(sorted_genes) - set(existing_genes)
if missing_genes:
    print(f"Warning: {len(missing_genes)} genes not found in adata: {list(missing_genes)[:5]}...")

# 高效提取表达矩阵
if issparse(adata.X):
    expression_matrix = adata[:, existing_genes].X.toarray()
else:
    expression_matrix = adata[:, existing_genes].X

# 创建DataFrame
expression_df = pd.DataFrame(
    expression_matrix,
    index=adata.obs_names,
    columns=existing_genes
)

# 添加leiden信息
expression_df['leiden'] = adata.obs['leiden'].values

# 5. 计算每个leiden簇的平均表达水平
print("Calculating mean expression per leiden cluster...")
mean_expression = expression_df.groupby('leiden')[existing_genes].mean()

# 6. 添加top5列
def get_top5_genes(row):
    """获取表达量最高的5个基因"""
    top5 = row.sort_values(ascending=False).head(5)
    return ', '.join(top5.index.tolist())

print("Adding top5 genes column...")
mean_expression['top5'] = mean_expression.apply(get_top5_genes, axis=1)

# 7. 创建基因分类行（用于添加到数据框顶部）
gene_classification_row = pd.DataFrame(
    [['Gene_Type'] + [gene_to_subtype[gene] for gene in existing_genes] + ['']],
    columns=['leiden'] + existing_genes + ['top5']
)

# 8. 合并分类行和平均表达数据
mean_expression_reset = mean_expression.reset_index()
final_df = pd.concat([gene_classification_row, mean_expression_reset], ignore_index=True)
# 9. 设置列顺序
final_df = final_df[['leiden'] + existing_genes + ['top5']]
# 10. 创建 leiden_ct_df (从adata中提取)
leiden_ct_df = adata.obs[['leiden', 'Celltype']].drop_duplicates().sort_values('leiden')
print(f"\nLeiden-Celltype mapping:\n{leiden_ct_df}")
# 11. 合并 final_df 和 leiden_ct_df
# 注意：需要保留分类行（leiden = 'Gene_Type'），然后为其他行添加Celltype信息
# 分离分类行和数据行
classification_row = final_df[final_df['leiden'] == 'Gene_Type'].copy()
data_rows = final_df[final_df['leiden'] != 'Gene_Type'].copy()
# 将leiden列转换为数值类型以便合并
# data_rows['leiden'] = pd.to_numeric(data_rows['leiden'])  # 不用，leiden_ct_df 中的也是 str
# 合并数据行与Celltype信息
data_rows_merged = data_rows.merge(leiden_ct_df, on='leiden', how='left')
# 检查是否有缺失的Celltype
missing_ct = data_rows_merged[data_rows_merged['Celltype'].isna()]
if len(missing_ct) > 0:
    print(f"\nWarning: {len(missing_ct)} leiden clusters missing Celltype information:")
    print(missing_ct[['leiden']])

# 重新排列列顺序：将Celltype放在leiden之后，top5之前
cols = ['leiden', 'Celltype'] + existing_genes + ['top5']
data_rows_merged = data_rows_merged[cols]
# 将分类行的Celltype列填充为''
classification_row['Celltype'] = ''

# 重新合并分类行和数据行，保持分类行在第一行
final_df_new = pd.concat([classification_row, data_rows_merged], ignore_index=True)
#### 再把 top5 DEGs的信息加上
Endo_leiden_degs = pd.read_csv('/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_100degs.tsv',sep='\t')
top5_degs_per_group = Endo_leiden_degs.groupby('group').head(5)
top5_degs_agg = top5_degs_per_group.groupby('group')['names'].agg(lambda x: ', '.join(x)).reset_index()
top5_degs_agg.columns = ['leiden', 'top5DEG']
classification_row_deg = pd.DataFrame({
    'leiden': ['Gene_Type'],
    'top5DEG': ['']
})
# 合并分类行和数据行
leiden_deg = pd.concat([classification_row_deg, top5_degs_agg], ignore_index=True)
final_df_new['leiden'] = final_df_new['leiden'].astype(str)
leiden_deg['leiden'] = leiden_deg['leiden'].astype(str)
final_df_new_with_degs = final_df_new.merge(leiden_deg, on='leiden', how='left')
cols = final_df_new_with_degs.columns.tolist()
# 找到 top5 列的位置
top5_idx = cols.index('leiden')
# 把 Celltype, top5, top5DEG 这几列都加入到 leiden列后面,便于比较
cols.insert(top5_idx + 1, cols.pop(cols.index('Celltype')))
cols.insert(top5_idx + 2, cols.pop(cols.index('top5')))
cols.insert(top5_idx + 3, cols.pop(cols.index('top5DEG')))
final_df_new_with_degs = final_df_new_with_degs[cols]
final_df_new_with_degs.to_csv('external_inputs/Endo_marker_all_exp.csv',sep=',',index=False)  # 然后自己去 WPS表格里进行高亮着色分块，隐藏滑动等操作
print("\n✅ Analysis complete!")



In [ ]:
########## 画 6Linage的code
genesets = ['T','B','Fibro', 'Endo','MoMac', 'Epi']
global_min = min([adata.obs[gs].min() for gs in genesets if gs in adata.obs])
global_max = max([adata.obs[gs].max() for gs in genesets if gs in adata.obs])
print(f"统一颜色范围: [{global_min:.3f}, {global_max:.3f}]")
# 绘制UMAP图，使用统一的vmin和vmax
axes = sc.pl.umap(adata, 
                  color=genesets,
                  color_map='viridis',
                  title=['T','B','Fibro', 'Endo','MoMac', 'Epi'],
                  frameon=False,
                  ncols=3,
                  size=3,
                  vmin=global_min,  # 统一最小值
                  vmax=global_max,  # 统一最大值
                  show=False,
                  legend_fontsize=18,
                  legend_fontoutline=2,
                  wspace=0.1)
for i, ax in enumerate(axes):
    ax.set_title(f'{ax.get_title()}', fontsize=20)  #   fontweight='bold'
    

# 调整图形大小
plt.gcf().set_size_inches(18, 9)
fig = plt.gcf()
# 创建一个用于colorbar的mappable对象
import matplotlib.cm as cm
from matplotlib.colors import Normalize
norm = Normalize(vmin=global_min, vmax=global_max)
sm = cm.ScalarMappable(norm=norm, cmap='viridis')
sm.set_array([])
plt.tight_layout()
fig_path = 'external_inputs/umap'
# plt.savefig(
#     fig_path+"6Lineage_umap.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )  ## pdf 存的好大，后续如果需要PDF文件需要优化下

plt.savefig(fig_path+"/6Lineage_umap.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=True)
plt.show()
plt.close()


In [ ]:
##### 画 Endo 中 subtype的code
genesets = ['SMC','Peri',  'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I', 'EndoMT-II']
axes = sc.pl.umap(adata, 
                  color=genesets,
                  color_map='viridis',
                  title=genesets,
                  frameon=False,
                  ncols=3,
                  size=3,
                  vmin=global_min,  # 统一最小值(6Lineage那里算了)
                  vmax=global_max,  # 统一最大值
                  show=False,
                  legend_fontsize=18,
                  legend_fontoutline=2,
                  wspace=0.1)
for i, ax in enumerate(axes):
    ax.set_title(f'{ax.get_title()}', fontsize=20)  #   fontweight='bold'


plt.gcf().set_size_inches(18, 12)
fig = plt.gcf()
plt.tight_layout()
fig_path = 'external_inputs/umap'
# plt.savefig(
#     fig_path+"Endo_all_umap.pdf",
#     dpi=150,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_path+"Endo_all_umap.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
##### 画 Leiden 和 其他 tissue 这些的代码 
plt.rcParams['figure.figsize'] = (6, 5)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='leiden',
    # palette=Nat_color_list,
    ax=ax,
    show=False,
    legend_loc='on data',   # 不用scanpy默认legend
    s=10,
    alpha=0.8
)
ax.set_title('Leiden', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig_dir='external_inputs/umap'
# plt.savefig(
#     fig_dir+"Endo_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Endo_leiden.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
##### 画 tissue 的代码 
plt.rcParams['figure.figsize'] = (10, 5)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='tissue',
    # palette=Nat_color_list,
    ax=ax,
    show=False,
    s=10,
    alpha=0.8
)
ax.set_title('Tissue', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig_dir='external_inputs/umap'
# plt.savefig(
#     fig_dir+"Endo_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Endo_tissue.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
######## 画每个leiden 中有多少细胞柱状图的code 
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']  # 解决中文显示（可选）
plt.rcParams['axes.unicode_minus'] = False

# 创建画布
fig, ax = plt.subplots(figsize=(16, 8))  # 画布大小可根据你的簇数调整

# 绘制柱状图
bars = ax.bar(
    x=df_sorted['leiden'],
    height=df_sorted['cell_count'],
    color='#1f77b4',  # 专业配色，可替换为你喜欢的颜色
    edgecolor='black',
    linewidth=0.8
)

# ✅ 关键：对数刻度y轴，完美解决大小簇差距过大
ax.set_yscale('log')
ax.set_ylim(bottom=df_sorted['cell_count'].min() * 0.8, top=df_sorted['cell_count'].max() * 1.1)

# ✅ 柱子顶部标注细胞数（千分位分隔，垂直显示避免拥挤）
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height * 1.02,  # 对数刻度下用比例偏移更合理
        f'{int(height):,}',  # 千分位分隔，如124,381
        ha='center',
        va='bottom',
        fontsize=8,
        rotation=90
    )

# ✅ 图表美化与标签
ax.set_title('Leiden CellNum(Log Scale)', fontsize=18, pad=20)
ax.set_xlabel('Leiden', fontsize=14, labelpad=15)
ax.set_ylabel('CellNum(Log Scale)', fontsize=14, labelpad=15)
ax.tick_params(axis='x', rotation=90, labelsize=10)  # x轴标签旋转，避免重叠
ax.tick_params(axis='y', labelsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.7)  # 横向网格线，方便读数
# 调整布局，避免标签截断
plt.tight_layout()
# ✅ 保存高清图片（300dpi满足顶刊要求）
plt.savefig(
    'external_inputs/leiden_cell_count_log.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()


In [ ]:
##### 画 Celltype_general 的代码 
plt.rcParams['figure.figsize'] = (7.5, 5)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata ,
    color='Celltype_general',
    # palette=Nat_color_list,
    ax=ax,
    show=False,
    s=10,
    alpha=0.8
)
ax.set_title('Celltype_general', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig_dir='external_inputs/umap'
# plt.savefig(
#     fig_dir+"Endo_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Endo_Celltype_general.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
############## 需要先基于 Leiden来进行画图  
score_columns = ['T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi']
# 2. 提取Leiden分组列（假设列名是'leiden'，如果不是请修改）
leiden_col = 'leiden'

# 3. 按Leiden分组计算每个细胞类型得分的均值（核心统计逻辑）
grouped_mean = adata.obs.groupby(leiden_col)[score_columns].mean().reset_index()
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")

# ====================== 配色：按你的细胞类型定制 ======================
# 适配你adata中的score列对应的细胞类型（去掉score_前缀匹配配色）
celltype_color_map = {
    'T': '#FF9933',        # 橙色
    'B': '#3399FF',       # 蓝色（对应你的score_Peri）
    'Fibro': '#9966CC',      # 紫色
    'Endo': '#FF6699',      # 粉色
    'MoMac': '#CCCC33',      # 黄色
    'Epi': '#66CC66'      # 绿色
    # 'EndoMT-I': '#33CCCC',   # 青色
    # 'EndoMT-II': '#CC99CC'   # 藕荷色
}

# 自动为每个score列分配对应颜色（从score_XXX中提取XXX匹配配色）
colors = []
for col in score_columns:
    colors.append(celltype_color_map.get(col, '#666666'))  # 找不到用灰色兜底

# ====================== 绘图布局 ======================
n_rows = 5
n_cols = int(np.ceil(n_leiden / n_rows))  # 按Leiden数量自适应列数
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 1.2*n_rows))
axes = axes.flatten()  # 展平子图数组方便遍历

# 总标题
fig.suptitle('6Lineage score in leiden',  
             fontsize=12, fontweight='bold', y=1.02)

# ====================== 绘制每个Leiden群的柱状图 ======================
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):  # 超出子图数量则停止
        break
    
    leiden = row.leiden
    ax = axes[i]
    
    # 提取当前Leiden群的所有细胞类型得分
    leiden_data = grouped_mean.loc[grouped_mean[leiden_col] == leiden, score_columns].iloc[0]
    
    # 设置柱子位置和间距（保持你原来的放大间距逻辑）
    x_pos = np.arange(len(score_columns)) * 2
    bars = ax.bar(x_pos, leiden_data.values, width=0.8, color=colors, 
                  edgecolor='black', linewidth=0.5)
    
    # 添加数值标签（适配不同数值大小的显示逻辑）
    max_score = grouped_mean[score_columns].values.max()
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        height = bar.get_height()
        
        # 调整标签位置：数值太小时贴柱子内部，正常时贴顶部
        if height < max_score * 0.05:
            text_y = height * 0.8
        else:
            text_y = height + max_score * 0.01
        
        # 格式化数值显示（根据大小自动调整精度）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加数值标签
        ax.text(x_pos[bar_idx], text_y, text,
                ha='center', va='bottom', fontsize=6, color='black', fontweight='normal')
    
    # 子图标题：仅显示Leiden编号
    ax.set_title(f'C_{leiden}', fontsize=6, fontweight='bold', pad=3)
    
    # 隐藏x轴刻度（避免拥挤）
    ax.set_xticks([])
    # 调整x轴范围适配间距
    ax.set_xlim(-0.6, x_pos.max() + 0.9)
    
    # 每行第一个子图显示y轴标签
    if i % n_cols == 0:
        ax.set_ylabel('Ave Score', fontsize=8)
    
    # 统一y轴范围
    ax.set_ylim([0, max_score * 1.15])
    
    # 添加网格线+美化边框
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#FFFFFF')  # 统一白色背景

# ====================== 隐藏多余子图 ======================
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# ====================== 图例：直接显示细胞类型名称 ======================
# 提取纯细胞类型名称作为图例标签（score_SMC→SMC）
legend_labels = score_columns
# 创建图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, facecolor=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(score_columns))]

# 添加图例到底部
fig.legend(legend_handles, legend_labels,
           loc='lower center', bbox_to_anchor=(0.5, 0.02),
           ncol=len(score_columns),  # 按细胞类型数量分列
           fontsize=7, title='Cell Type', title_fontsize=8,
           frameon=True, fancybox=True, framealpha=0.9)

# ====================== 调整布局 ======================
plt.tight_layout()
plt.subplots_adjust(top=0.92, bottom=0.15, left=0.05, right=0.95, hspace=0.5, wspace=0.5)
fig_dir='external_inputs/annotation'
# plt.savefig(
#     fig_dir+"Endo_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"6Lineage_score.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()



In [ ]:
############## 需要先基于 Leiden来进行画图, 上面画好了 6个Lineage的，避免有些有偏差，再画下 8个 subtype的
score_columns = ['MCs', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT']   #  ['MCs', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I','EndoMT-II']
# 2. 提取Leiden分组列（假设列名是'leiden'，如果不是请修改）
leiden_col = 'leiden'

# 3. 按Leiden分组计算每个细胞类型得分的均值（核心统计逻辑）
grouped_mean = adata.obs.groupby(leiden_col)[score_columns].mean().reset_index()
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")

# ====================== 配色：按你的细胞类型定制 ======================
# 适配你adata中的score列对应的细胞类型（去掉score_前缀匹配配色）
celltype_color_map = {
    'MCs': '#FF9933',        # 橙色
  #  'Peri': '#3399FF',       # 蓝色（对应你的score_Peri）
    'LEC': '#9966CC',      # 紫色
    'VenEC': '#FF6699',      # 粉色
    'CapEC': '#CCCC33',      # 黄色
    'ArtEC': '#66CC66',      # 绿色
    'EndoMT':'#33CCCC',   # 青色
   #  'EndoMT-II': '#CC99CC'   # 藕荷色
}

# 自动为每个score列分配对应颜色（从score_XXX中提取XXX匹配配色）
colors = []
for col in score_columns:
    colors.append(celltype_color_map.get(col, '#666666'))  # 找不到用灰色兜底

# ====================== 绘图布局 ======================
n_rows = 5
n_cols = int(np.ceil(n_leiden / n_rows))  # 按Leiden数量自适应列数
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.3*n_cols, 1.2*n_rows))
axes = axes.flatten()  # 展平子图数组方便遍历

# 总标题
fig.suptitle('subType score in leiden',  
             fontsize=12, fontweight='bold', y=1.02)

# ====================== 绘制每个Leiden群的柱状图 ======================
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):  # 超出子图数量则停止
        break
    
    leiden = row.leiden
    ax = axes[i]
    
    # 提取当前Leiden群的所有细胞类型得分
    leiden_data = grouped_mean.loc[grouped_mean[leiden_col] == leiden, score_columns].iloc[0]
    
    # 设置柱子位置和间距（保持你原来的放大间距逻辑）
    x_pos = np.arange(len(score_columns)) * 2
    bars = ax.bar(x_pos, leiden_data.values, width=0.8, color=colors, 
                  edgecolor='black', linewidth=0.5)
    
    # 添加数值标签（适配不同数值大小的显示逻辑）
    max_score = grouped_mean[score_columns].values.max()
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        height = bar.get_height()
        
        # 调整标签位置：数值太小时贴柱子内部，正常时贴顶部
        if height < max_score * 0.05:
            text_y = height*0.8
        else:
            text_y = height + max_score * 0.01
        
        # 格式化数值显示（根据大小自动调整精度）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加数值标签
        # ax.text(x_pos[bar_idx], text_y, text,
        #         ha='center', va='bottom', fontsize=6, color='black', fontweight='normal')
        ax.text(x_pos[bar_idx], text_y, text,
                ha='center',           # 水平居中
                va='bottom' if height >= max_score * 0.05 else 'top',  # 根据位置调整垂直对齐
                rotation=90,           # 旋转90度，竖直摆放
                fontsize=5,            # 字体稍微调小一点以适应竖直排列
                color='black', 
                fontweight='normal')
    
    # 子图标题：仅显示Leiden编号
    ax.set_title(f'C_{leiden}', fontsize=6, fontweight='bold', pad=3)
    
    # 隐藏x轴刻度（避免拥挤）
    ax.set_xticks([])
    # 调整x轴范围适配间距
    ax.set_xlim(-0.6, x_pos.max() + 0.9)
    
    # 每行第一个子图显示y轴标签
    if i % n_cols == 0:
        ax.set_ylabel('Ave Score', fontsize=8)
    
    # 统一y轴范围
    ax.set_ylim([0, max_score * 1.15])
    
    # 添加网格线+美化边框
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#FFFFFF')  # 统一白色背景

# ====================== 隐藏多余子图 ======================
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# ====================== 图例：直接显示细胞类型名称 ======================
# 提取纯细胞类型名称作为图例标签（score_SMC→SMC）
legend_labels = score_columns
# 创建图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, facecolor=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(score_columns))]

# 添加图例到底部
fig.legend(legend_handles, legend_labels,
           loc='lower center', bbox_to_anchor=(0.5, 0.02),
           ncol=len(score_columns),  # 按细胞类型数量分列
           fontsize=7, title='Cell Type', title_fontsize=8,
           frameon=True, fancybox=True, framealpha=0.9)

# ====================== 调整布局 ======================
plt.tight_layout()
plt.subplots_adjust(top=0.92, bottom=0.15, left=0.05, right=0.95, hspace=0.5, wspace=0.5)
fig_dir='external_inputs/annotation'
# plt.savefig(
#     fig_dir+"Endo_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"subEndo_score.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
####### 为了注释是 leiden 是 tissue_specific 还是 tissue_pan 的，画个 dominant tissue 的图
import pandas as pd
import scanpy as sc
import seaborn as sns
# 假设 adata 是您的单细胞对象，且包含 'leiden' 和 'tissue' 列
# 如果列名不同（如 'kissue'），请替换下方代码中的 'tissue'

# 1. 统计每个 leiden 中不同 tissue 的计数
meta_data = adata.obs
count_df = meta_data.groupby(['leiden', 'tissue']).size().unstack(fill_value=0)                 #    adata.obs.groupby(['leiden', 'tissue']).size().unstack(fill_value=0)

# 2. 计算每个 leiden 中 tissue 的占比（行内比例）
percentage_df = count_df.div(count_df.sum(axis=1), axis=0) * 100  # 转换为百分比
# 3. 提取每个 leiden 中占比最高的 tissue 及其比例
max_tissue = percentage_df.idxmax(axis=1)
max_percentage = percentage_df.max(axis=1)

# 4. 构建结果 DataFrame
dominant_tissue_percent = pd.DataFrame({
    'leiden': max_tissue.index,
    'dominant': max_tissue.values,
    'percentage': max_percentage.values.round(2)  # 保留两位小数
}).reset_index(drop=True)


# 假设 df1 是之前生成的结果，包含列：'leiden', 'dominant_tissue', 'percentage'
# 设置图形风格
sns.set_style("whitegrid")
plt.figure(figsize=(18, 8))

df1 = dominant_tissue_percent.sort_values('percentage')
leiden_order = df1.sort_values('percentage')['leiden'].tolist()
# 绘制柱状图（颜色按 dominant_tissue 自动分配）
ax = sns.barplot(
    data=df1,
    x="leiden",
    y="percentage",
    order=leiden_order,
    color="dominant",
   # hue="dominant",  # 按 tissue 类型着色
    # palette="viridis",      # 使用 viridis 调色板（可替换为 'tab20' 等）
    dodge=False            # 确保柱子不分开
)

# 添加标题和标签
plt.title("Dominant Tissue Percentage", fontsize=14)
plt.xlabel("Leiden", fontsize=12)
plt.ylabel("Percentage(%)", fontsize=12)

# 调整图例位置和样式
plt.legend(
    title="Dominant Tissue",
    bbox_to_anchor=(1, 1),  # 将图例放在图形右侧
    loc="upper left",
    frameon=False,
    fontsize=13
)

# 显示百分比标签（可选）
for p in ax.patches:
    height = p.get_height()
    ax.text(
        p.get_x() + p.get_width() / 2.,
        height + 1,
        f"{height:.1f}",
        ha="center",
        rotation=90,
        fontsize=12
    )

# 自动调整布局避免标签重叠
plt.tight_layout()
fig_dir='external_inputs/annotation'
# plt.savefig(
#     fig_dir+"Endo_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Dominant_tissue.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()



In [ ]:
######## 对 adata 中的不同 leiden进行 subtype 注释, 需要先画下每个leiden 中6+8 个signature score的图

cluster_to_celltype = {  
  '0': 'VenEC_CLU_pan', 
  '1': 'Epi_TAGLN_pan',  # 改 Epi
  '2': 'VenEC_TM4SF1_pan',  
  '3': 'VenEC_TAGLN_prostate', 
  '4': 'T_CD3E_pan', 
  '5': 'EndoMT-I_CFD_bone marrow',    # AP很高,Stalk也不低
  '6': 'Mono/Macro_S100A8_bone marrow',
  '7': 'CapEC_FLT1_pan',    
  '8': 'VenEC_PTN_pan',    ## 特定病理或发育状态下与内皮细胞功能密切相关的分泌性因子,促血管生成
  '9': 'VenEC_DCN_breast',  
  '10': 'VenEC_FLT1_pan',  
  '11': 'CapEC_TM4SF1_breast',  # CapEC (0.6159),	ArtEC ( 0.6213) 
  '12': 'CapEC_TM4SF1_pan',  
  '13': 'CapEC_TM4SF1_pan',   
  '14': 'Mono/Macro_NDUFA4L2_skin', 
  '15': 'Mono/Macro_LST1_pan',  # 改:EndoMT-II
  '16': 'VenEC_TM4SF1_breast',  
  '17': 'Epi_NOTCH3_pan',  # 改:Epi
  '18': 'VenEC_CLU_eye', 
  '19': 'EndoMT-I_GNG11_vessel', 
  '20': 'CapEC_GNG11_pan', 
  '21': 'VenEC_CFD_pan',  # 改:EndoMT-II 
  '22': 'EndoMT-I_COL1A1_pan',   
  '23': 'Epi_TAGLN_pancreas',  # 改:Epi 还是 VenEC 
  '24': 'ArtEC_CXCL12_pan',   
  '25': 'VenEC_CFD_pan',  
  '26': 'CapEC_FLT1_brain',   
  '27': 'T_CD3D_pan',   # EndoMT-I_o 低一点
  '28': 'Mono/Macro_MS4A7_pan',  # 但 AP 最高 
  '29': 'Epi_EPCAM_pan',   
  '30': 'VenEC_CLU_eye', 
  '31': 'SMC_TPM2_pan',  
  '32': 'EndoMT-I_TPM2_pan', # SMC低一点
  '33': 'Epi_DCN_breast', # VenEC 低一点
  '34': 'B_CD38_pan',  
  '35': 'VenEC_EFEMP1_pan',  # VenEC 高一点
  '36': 'Epi_RGCC_lung',   # 但RGCC其实是CapEC的marker
  '37': 'Epi_TPM2_pan',  # Epi 高一点
  '38': 'VenEC_FLT1_uterus',  
  '39': 'Epi_TAGLN_pan',   # VenEC 高一点
  '40': 'CapEC_GNG11_pan',  
  '41': 'EndoMT-II_TGFBI_pan',  # 最后那列SS注释的
  '42': 'SMC_CFD_pan', 
  '43': 'VenEC_TAGLN_pan',  ## TAGLN其实是 SMC的marker，但在Peri中也表达
  '44': 'VenEC_TAGLN_pan',  # 
  '45': 'VenEC_THY1_pan', # 各个SS表达都很低,    但 THY1 淋巴管内皮细胞 (Lymphatic Endothelial Cells, LEC) 中的表达水平远高于血液血管内皮细胞, 
  '46': 'Epi_COL1A1_pan', # 
  '47': 'EndoMT-I_TM4SF1_pan',  
  '48': 'VenEC_EPCAM_pan',  
  '49': 'VenEC_CLU_eye', #  
  '50': 'LEC_EFEMP1_skin', # 
  '51': 'ArtEC_ARL15_pan',  
  '52': 'ArtEC_HSPG2_brain', 
  '53': 'B_CDK14_skin',
  '54': 'EndoMT-I_TM4SF1_lymph' 
}  


cluster_to_celltype = {  
  '0': 'VenEC_CLU_pan', 
  '1': 'Epi_TAGLN_pan',  # 改 Epi 
  '2': 'CapEC_TM4SF1_pan',  
  '3': 'CapEC_TAGLN_prostate', 
  '4': 'T_CD3E_pan', 
  '5': 'EndoMT_CFD_bone marrow',    # AP很高,Stalk也不低
  '6': 'Mono/Macro_S100A8_bone marrow',
  '7': 'ArtEC_FLT1_pan',    
  '8': 'EndoMT_PTN_pan',    ## 特定病理或发育状态下与内皮细胞功能密切相关的分泌性因子,促血管生成
  '9': 'CapEC_DCN_breast',  
  '10': 'VenEC_FLT1_pan',  
  '11': 'CapEC_TM4SF1_breast',  # CapEC (0.6159),	ArtEC ( 0.6213) 
  '12': 'CapEC_TM4SF1_pan',  
  '13': 'CapEC_TM4SF1_pan',   
  '14': 'Mono/Macro_NDUFA4L2_skin', 
  '15': 'Mono/Macro_LST1_pan',  # 改:EndoMT
  '16': 'VenEC_TM4SF1_breast',  
  '17': 'Mono/Macro_NOTCH3_pan',  # 改:Epi
  '18': 'VenEC_CLU_eye', 
  '19': 'EndoMT_GNG11_vessel', 
  '20': 'CapEC_GNG11_pan', 
  '21': 'EndoMT_CFD_pan',  # 改:EndoMT
  '22': 'EndoMT_COL1A1_pan',   
  '23': 'VenEC_TAGLN_pancreas',  # 改:Epi 还是 VenEC  
  '24': 'ArtEC_CXCL12_pan',   
  '25': 'VenEC_CFD_pan',  
  '26': 'ArtEC_FLT1_brain',   
  '27': 'EndoMT_CD3D_pan',   # EndoMT-I_o 低一点
  '28': 'EndoMT_VIM_pan',  # 但 AP 最高 
  '29': 'Epi_EPCAM_pan',   
  '30': 'VenEC_CLU_eye', 
  '31': 'MCs_TPM2_pan',  
  '32': 'EndoMT_TPM2_pan', # EndoMT-I 低一点
  '33': 'VenEC_DCN_breast', # VenEC 低一点
  '34': 'B_CD38_pan',  
  '35': 'LEC_EFEMP1_pan',  # VenEC 高一点
  '36': 'Epi_RGCC_lung',   # 但RGCC其实是CapEC的marker
  '37': 'EndoMT_TPM2_pan',  # Epi 高一点
  '38': 'CapEC_FLT1_uterus',  
  '39': 'CapEC_TAGLN_pan',   # VenEC 高一点
  '40': 'CapEC_GNG11_pan',  
  '41': 'EndoMT_TGFBI_pan',  # 最后那列SS注释的
  '42': 'EndoMT_CFD_pan', 
  '43': 'EndoMT_TAGLN_pan',  ## TAGLN其实是 SMC的marker，但在Peri中也表达  
  '44': 'VenEC_TAGLN_pan',  # 
  '45': 'ArtEC_THY1_pan', # 各个SS表达都很低,    但 THY1 淋巴管内皮细胞 (Lymphatic Endothelial Cells, LEC) 中的表达水平远高于血液血管内皮细胞,  
  '46': 'Epi_GNG11_pan', #  GNG11 是 LEC marker
  '47': 'EndoMT_TM4SF1_pan',  
  '48': 'VenEC_EPCAM_pan',  
  '49': 'EndoMT_CLU_eye', #  
  '50': 'EndoMT_EFEMP1_skin', 
  '51': 'ArtEC_ARL15_pan',  
  '52': 'ArtEC_HSPG2_brain',   # 
  '53': 'B_CDK14_skin',
  '54': 'EndoMT_TM4SF1_lymph'  
}  


adata.obs['Celltype'] = adata.obs['leiden'].map(cluster_to_celltype)
unmapped_clusters = set(adata.obs['leiden']) - set(cluster_to_celltype.keys())
if unmapped_clusters:
    print(f"未定义的聚类: {unmapped_clusters}") 
    adata.obs['Celltype'] = adata.obs['leiden'].map(lambda x: cluster_to_celltype.get(x, 'Unknown'))

adata.obs['Celltype_sub'] = adata.obs['Celltype'].str.split('_').str[:-1].str.join('_')
adata.obs['Celltype_general'] = adata.obs['Celltype_sub'].str.split('_').str[:-1].str.join('_')
adata.write('/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn.h5ad')


In [ ]:
score_df = adata.obs.groupby('leiden')[signature_cols].mean().reset_index()
score_df.to_csv('external_inputs/score_by_leiden.csv',sep=',',index=False)



In [ ]:
signature_cols = ['MS_Fib_deg', 'MS_Endo_deg','Fib_deg', 'Endo_deg', 'AP', 'Tip', 'Stalk' ,'T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi',
       'SMC', 'Peri', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I',
       'EndoMT-II', 'Fib_deg', 'Endo_deg', 'Celltype_general', 'Celltype',
       'Celltype_sub', 'SMC1', 'Peri1', 'LEC1', 'VenEC1', 'CapEC1', 'ArtEC1',
       'SMC_o', 'Peri_o', 'LEC_o', 'VenEC_o', 'CapEC_o', 'ArtEC_o',
       'EndoMT-I_o', 'EndoMT-II_o',  'SMC_1', 'Peri_1',
       'LEC_1', 'VenEC_1', 'CapEC_1', 'ArtEC_1', 'EndoMT-I_1', 'EndoMT-II_1']
score_df = adata.obs.groupby('leiden')[signature_cols].mean().reset_index()
score_df.to_csv('external_inputs/score_by_leiden.csv',sep=',',index=False)





In [ ]:

def highlight_max_rowwise(row):
    max_val = row[signature_cols].max()
    return ['color: red; font-weight: bold' if v == max_val and pd.notna(v) else ''  for v in row[signature_cols]]
    

In [ ]:
import pandas as pd
import numpy as np
import openpyxl 

# ['MS_Fib_deg', 'MS_Endo_deg','Fib_deg', 'Endo_deg', 'AP', 'Tip', 'Stalk' ,'T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi',
#        'SMC', 'Peri', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I', 'EndoMT-II',  'SMC1', 'Peri1', 'LEC1', 'VenEC1', 'CapEC1', 'ArtEC1',
#        'SMC_o', 'Peri_o', 'LEC_o', 'VenEC_o', 'CapEC_o', 'ArtEC_o', 'EndoMT-I_o', 'EndoMT-II_o']

score_df = pd.read_csv('external_inputs/score_by_leiden.csv',sep=',')
signature_cols = ['Fib_deg', 'Endo_deg', 'AP', 'Tip', 'Stalk' ,'T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi',
       'SMC', 'Peri', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I',
       'EndoMT-II',  'SMC1', 'Peri1', 'LEC1', 'VenEC1', 'CapEC1', 'ArtEC1',
       'SMC_o', 'Peri_o', 'LEC_o', 'VenEC_o', 'CapEC_o', 'ArtEC_o',
       'EndoMT-I_o', 'EndoMT-II_o']
styled_score_df = score_df[signature_cols].style.apply(highlight_max_rowwise, axis=1)   ### 截取某些行的话就用  score_df[signature_cols].tail(25) 或者 score_df[signature_cols].head(30)

# 4. 设置更美观的显示格式（可选：设置数值精度）
styled_score_df = styled_score_df.format({col: '{:.4f}' for col in signature_cols})
styled_score_df


In [ ]:
##### 

In [ ]:
###### 画 Endothelial marker list的图，一个gene 一个 UMAP
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

union_list = [gene for gene in Endo_marker_union if gene in adata.var_names]
n_cols = 10
n_rows =  4
figsize = (26, 8)  # 总画布大小

# 创建图形
fig = plt.figure(figsize=figsize)

# 使用 GridSpec 创建子图网格
gs = GridSpec(n_rows, n_cols, figure=fig, wspace=0.3, hspace=0.4)

# 对每个基因绘制 UMAP
for idx, gene in enumerate(union_list):
    row = idx // n_cols
    col = idx % n_cols
    ax = fig.add_subplot(gs[row, col])
    # 绘制 UMAP
    sc.pl.umap(adata, color=gene, ax=ax, show=False, size=2,
               title=gene, frameon=True,
               legend_loc='none')  # 隐藏颜色条，减少拥挤
    
    # 自定义标题和标签
    ax.set_title(gene, fontsize=8, fontweight='bold', pad=2)
    ax.set_xlabel('', fontsize=6)
    ax.set_ylabel('', fontsize=6)
    ax.tick_params(labelsize=5)
    
    # 移除坐标轴标签（可选，使图更简洁）
    ax.set_xlabel('')
    ax.set_ylabel('')

# 隐藏多余的子图
for idx in range(len(union_list), n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    ax = fig.add_subplot(gs[row, col])
    ax.axis('off')

# 添加总标题
fig.suptitle('Endothelial marker gene umap', 
             fontsize=14, y=0.98)

# 调整布局
plt.tight_layout()
plt.subplots_adjust(top=0.96, bottom=0.02, left=0.02, right=0.98)
fig_dir='external_inputs/umap'
fig_name = 'Endo_markers.png'
plt.savefig(fig_dir+fig_name, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()


In [ ]:
sub_leiden =['4', '6', '15', '27', '28', '29', '34', '48', '53']
adata_sub = adata[~adata.obs['leiden'].isin(sub_leiden), :]  # 去掉那几个不是 Endothelial 的cluster
adata = adata_sub


In [ ]:
######## 然后对 leiden 进行重命名
leiden_unique = adata.obs['leiden'].unique()
leiden_numeric = np.sort([int(x) for x in leiden_unique]).tolist()  # 转为整数并排序
leiden_new = np.arange(0, len(leiden_numeric)).tolist()
if len(leiden_numeric) != len(leiden_new):
    raise ValueError("l1 和 l2 的长度必须相同")

old_to_new = {str(key): str(value) for key, value in zip(leiden_numeric, leiden_new)}
adata.obs['leiden'] = adata.obs['leiden'].map(old_to_new)
adata.write('/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo.h5ad')


In [ ]:
# sorted(dir())



In [ ]:
############ 画每个 tissue 中有多少 sample 和多少 cell的图
# 提取tissue和GSM列
tissue_gsm_df = adata.obs[["tissue", "sample"]].copy()

#统计各组织的细胞数和原始GSM数
tissue_cell_count = tissue_gsm_df.groupby("tissue").size().reset_index(name="cell_count")
tissue_gsm_count = tissue_gsm_df.groupby("tissue")["sample"].nunique().reset_index(name="gsm_count")

# 合并数据
tissue_stats = pd.merge(tissue_cell_count, tissue_gsm_count, on="tissue", how="outer")
tissue_stats = tissue_stats.dropna(subset=["cell_count", "gsm_count"]).reset_index(drop=True)
# 转换数据类型
tissue_stats["cell_count"] = tissue_stats["cell_count"].astype(int)
tissue_stats["gsm_count"] = tissue_stats["gsm_count"].astype(int)

# 按细胞数降序排列
custom_tissue_order = tissue_stats.sort_values("cell_count", ascending=False)["tissue"].tolist()
tissue_stats = tissue_stats[tissue_stats["tissue"].isin(custom_tissue_order)].copy()
tissue_stats["tissue"] = pd.Categorical(tissue_stats["tissue"], categories=custom_tissue_order, ordered=True)
tissue_stats = tissue_stats.sort_values("tissue").reset_index(drop=True)

#GSM负向高度放大500倍
zoom_factor = 500  # 固定放大倍数
tissue_stats["gsm_count_neg_zoom"] = -tissue_stats["gsm_count"] * zoom_factor  # 负向显示

# 打印统计结果
print("各组织的细胞数（降序）、实际GSM数与放大后柱子高度：")
print(tissue_stats[["tissue", "cell_count", "gsm_count", "gsm_count_neg_zoom"]])
print(f"\nGSM放大倍数：{zoom_factor}倍（GSM柱子高度=实际GSM数×{zoom_factor}）")

plt.figure(figsize=(14, 8))  # 图表尺寸
# 颜色
cell_color = "#2E86AB"  # 细胞数（蓝色系）
gsm_color = "#A23B72"   # GSM数（紫红色系）

# 细胞数（向上）
bars_cell = plt.bar(
    x=tissue_stats["tissue"],  # x轴为组织类别（细胞数降序）
    height=tissue_stats["cell_count"],  # y轴为细胞数
    color=cell_color,
    alpha=0.8,
    label="Cell Number"  # 图例
)

# GSM数（向下，放大500倍）
bars_gsm = plt.bar(
    x=tissue_stats["tissue"],  # x轴与细胞数对齐
    height=tissue_stats["gsm_count_neg_zoom"],  # y轴为负向放大后的GSM高度
    color=gsm_color,
    alpha=0.8,
    label="Sample Number"  # 图例
)


# 修改y轴刻度（负向显示实际GSM数，正向显示原始细胞数）
ax = plt.gca()  # 获取当前坐标轴对象
y_ticks = ax.get_yticks()  # 自动生成的y轴刻度值

custom_y_labels = []
for tick in y_ticks:
    if tick < 0:  # 选中负向刻度
        actual_gsm = int(tick / (-zoom_factor))  # 还原为实际GSM数（刻度值 ÷ (-放大倍数)）
        custom_y_labels.append(str(actual_gsm))
    elif tick == 0:  # 0刻度：保持“0”
        custom_y_labels.append("0")
    else:  # 选中正向刻度：保留原始数值并添加千分位
        custom_y_labels.append(f"{int(tick):,}")

# 应用自定义刻度标签
ax.set_yticklabels(custom_y_labels, fontsize=9)
plt.xticks(rotation=-45, ha='left', fontsize=9)  # 保持x轴标签旋转
# 细胞数标注（显示各组织的实际细胞数，带千分位）
for bar, count in zip(bars_cell, tissue_stats["cell_count"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,  # 标注水平居中（柱子中点）
        bar.get_height() + tissue_stats["cell_count"].max() * 0.01,  # 标注在柱子顶部上方（避免重叠）
        s=f"{count:,}",  # 实际细胞数
        ha="center", va="bottom",  # 对齐方式
        fontsize=9,
        rotation=90,
        fontweight="bold"
    )

# GSM数标注
for bar, count in zip(bars_gsm, tissue_stats["gsm_count"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,  # 标注水平居中
        bar.get_height() - (tissue_stats["gsm_count"].max() * zoom_factor) * 0.02,  # 标注在柱子底部下方
        s=f"{count}",  # 实际GSM数
        ha="center", va="top",  # 对齐方式
        fontsize=9,
        fontweight="bold",
        color=gsm_color  # 标注颜色
    )

plt.xlabel("Tissue Type ", fontsize=12, labelpad=10)  
plt.ylabel("Count", fontsize=11, labelpad=10)  
plt.title("Distribution of CellNum and SampleNum of Different Tissus", fontsize=14, pad=20)  

plt.legend(loc="upper center", fontsize=10, bbox_to_anchor=(0.5, 0.7), ncol=2)  # 图例位置
plt.axhline(y=0, color="black", linestyle="-", linewidth=2)  # 加粗0刻度线
# 调整布局
plt.tight_layout()
fig_dir='external_inputs/OnlyEndo'
plt.savefig(fig_dir+"/Endo_tissue_sam_cell_Num.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show() 



In [ ]:
########### 计算之前没计算的那 3个 functional signature list的score
def calculate_signature_scores_raw(adata, signature_dict, use_raw=True):
    """
    使用raw count数据计算signature scores
    """
    # 确保使用raw counts
    if use_raw and adata.raw is not None:
        print("使用raw count数据...")
        count_data = adata.raw.X
        var_names = adata.raw.var_names
    else:
        print("使用当前数据...")
        count_data = adata.X
        var_names = adata.var_names
    from scipy.sparse import issparse
    signature_scores = {}
    available_genes_info = {}
    for signature_name, genes in signature_dict.items():
        # 检查哪些基因在数据中存在
        available_genes = [gene for gene in genes if gene in var_names]
        missing_genes = set(genes) - set(available_genes)
        available_genes_info[signature_name] = {
            'available': available_genes,
            'missing': missing_genes,
            'available_count': len(available_genes)
        }
        if len(available_genes) == 0:
            print(f"错误: {signature_name} 没有可用的基因")
            signature_scores[signature_name] = np.zeros(adata.n_obs)
            continue
        if len(missing_genes) > 0:
            print(f"警告: {signature_name} 中以下基因不存在: {missing_genes}")
        # 提取基因表达矩阵
        gene_indices = [np.where(var_names == gene)[0][0] for gene in available_genes]
        if issparse(count_data):
            expr_matrix = count_data[:, gene_indices].toarray()
        else:
            expr_matrix = count_data[:, gene_indices]
        # 方法1: 计算平均表达值 (使用raw counts)
        mean_scores = np.mean(expr_matrix, axis=1)
        # 方法2: 计算表达百分比 (表达该signature中至少一个基因的细胞)
        expr_percentage = np.mean(expr_matrix > 0, axis=1)
        # 综合score: 结合平均表达和表达百分比
        # 对平均表达进行log1p转换以避免极端值
        mean_scores_log = np.log1p(mean_scores)
        combined_score = mean_scores_log * expr_percentage
        signature_scores[signature_name] = combined_score
        adata.obs[f"score_{signature_name}"] = combined_score
    # 打印可用基因统计
    print("\n各亚型可用基因统计:")
    for subtype, info in available_genes_info.items():
        print(f"{subtype}: {info['available_count']}/{len(signature_dict[subtype])} 个基因可用")
    return adata
    

In [ ]:

adata = calculate_signature_scores_raw(adata, score_dict)
adata.write('/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo.h5ad')


In [ ]:
####### 画 tip, antigen_pre, stalk 的二维散点图
score_cols =  list(score_dict.keys())
# 提取：leiden（用于分组）、Celltype_general（用于标注）、各基因集得分
leiden_score_df = adata.obs[["leiden", "Celltype_general"] + score_cols].copy()
# --------------------------
# 关键修正：agg()直接接收字典作为位置参数，无需**解包
# --------------------------
# 1. 构建“列名-聚合函数”字典（score列取均值，Celltype_general取第一个值）
agg_dict = {col: "mean" for col in score_cols}  # 所有score列用mean聚合
agg_dict["Celltype_general"] = "first"  # Celltype_general用first聚合（每个leiden组唯一）
# 2. 直接将agg_dict作为位置参数传给agg()，避免**解包导致的识别问题
leiden_avg_score = leiden_score_df.groupby("leiden").agg(
    agg_dict  # 核心修改：去掉**，直接传字典
).reset_index()

# 【优化】将Celltype_general转为数值编码（用于颜色映射，相同Celltype颜色一致）
leiden_avg_score["Celltype_code"] = leiden_avg_score["Celltype_general"].factorize()[0]

# 定义3个基因集得分列（保持原逻辑不变）
score1, score2, score3 = score_cols  # 对应Antigen_pre、Tip、Stalk（按score_dict顺序）
score1_name =score1   #       score1.replace("score_", "")
score2_name =  score2           
score3_name = score3

# 创建1行3列的子图布局（保持原逻辑不变）
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# --------------------------
# 子图1：score1（x） vs score2（y）
# --------------------------
ax1.scatter(
    leiden_avg_score[score1],
    leiden_avg_score[score2],
    c=leiden_avg_score["Celltype_code"],
    cmap="tab10",  # 分类色板，适合区分Celltype
    s=100,
    alpha=0.8
)
# 标注Celltype名称
for _, row in leiden_avg_score.iterrows():
    ax1.text(
        row[score1], row[score2], 
        s=row["Celltype_general"],
        fontsize=8, ha="center", va="bottom",
        fontweight="bold"
    )

ax1.set_xlabel(score1_name, fontsize=11)
ax1.set_ylabel(score2_name, fontsize=11)
ax1.set_title(f"{score1_name} vs {score2_name}\n(Celltype)", fontsize=12, pad=10)
ax1.grid(alpha=0.3)

# --------------------------
# 子图2：score1（x） vs score3（y）
# --------------------------
ax2.scatter(
    leiden_avg_score[score1],
    leiden_avg_score[score3],
    c=leiden_avg_score["Celltype_code"],
    cmap="tab10",
    s=100,
    alpha=0.8
)

for _, row in leiden_avg_score.iterrows():
    ax2.text(
        row[score1], row[score3], 
        s=row["Celltype_general"],
        fontsize=8, ha="center", va="bottom",
        fontweight="bold"
    )

ax2.set_xlabel(score1_name, fontsize=11)
ax2.set_ylabel(score3_name, fontsize=11)
ax2.set_title(f"{score1_name} vs {score3_name}\n(Celltype)", fontsize=12, pad=10)
ax2.grid(alpha=0.3)
# --------------------------
# 子图3：score2（x） vs score3（y） 
# --------------------------
ax3.scatter(
    leiden_avg_score[score2],
    leiden_avg_score[score3],
    c=leiden_avg_score["Celltype_code"],
    cmap="tab10",
    s=100,
    alpha=0.8
)
for _, row in leiden_avg_score.iterrows():
    ax3.text(
        row[score2], row[score3], 
        s=row["Celltype_general"],
        fontsize=8, ha="center", va="bottom",
        fontweight="bold" )

ax3.set_xlabel(score2_name, fontsize=11)
ax3.set_ylabel(score3_name, fontsize=11)
ax3.set_title(f"{score2_name} vs {score3_name}\n(Celltype)", fontsize=12, pad=10)
ax3.grid(alpha=0.3)
# 调整子图间距，避免标题/标注被截断
plt.tight_layout(rect=[0, 0.03, 0.97, 0.95])
fig_dir='external_inputs/OnlyEndo'
plt.savefig(fig_dir+"/Endo_tip_ap_stalk_signature_scatterplot.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()


In [ ]:
######## 上面的是 2D的图，现在画一个 3D的图
########## 先准备数据
score_cols =  list(score_dict.keys())
# 提取：leiden（用于分组）、Celltype_general（用于标注）、各基因集得分
leiden_score_df = adata.obs[["leiden", "Celltype_general"] + score_cols].copy()
# 按Leiden簇分组，计算每个基因集得分的均值（与第一段完全一致的均值逻辑）
leiden_avg_score = leiden_score_df.groupby("leiden").agg({col: "mean" for col in score_cols}).reset_index()

# 关联细胞类型
# 提取Leiden簇与细胞类型的对应关系
leiden_celltype =adata.obs[['leiden', "Celltype_general"]].drop_duplicates(subset="Celltype_general").reset_index(drop=True)  # adata.obs[["leiden", "Celltype"]].drop_duplicates(subset="leiden").reset_index(drop=True)

# 合并“簇均值得分”和“细胞类型”（确保每个Leiden簇对应唯一得分和细胞类型）
leiden_score_with_celltype = leiden_avg_score.merge(
    leiden_celltype, 
    on="Celltype_general",  # 按Leiden簇编号关联
    how="inner"   # 只保留两边都存在的簇（避免空值）
)


import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# 创建图形
fig = plt.figure(figsize=(18, 14))
ax = fig.add_subplot(111, projection='3d')

# 提取数据
x = leiden_avg_score['AP'].values
y = leiden_avg_score['Tip'].values
z = leiden_avg_score['Stalk'].values
celltypes = leiden_avg_score['Celltype_general'].values

# 生成颜色
n_colors = len(leiden_avg_score)
color_list = plt.cm.tab20(np.linspace(0, 1, min(20, n_colors))).tolist()
if n_colors > 20:
    additional_colors = plt.cm.Set3(np.linspace(0, 1, n_colors - 20)).tolist()
    color_list.extend(additional_colors)

# 绘制散点图 - 使用不同大小区分密集区域
# 计算每个点与其他点的平均距离，密集区域的点画小一点
from scipy.spatial import distance

points = np.column_stack([x, y, z])
avg_distances = []
for i in range(len(points)):
    dists = distance.cdist([points[i]], points)[0]
    avg_distances.append(np.mean(dists))

# 根据密集程度调整点大小
point_sizes = np.interp(avg_distances, 
                       [min(avg_distances), max(avg_distances)], 
                       [80, 150])

scatter = ax.scatter(x, y, z, 
                    c=color_list,
                    s=point_sizes,
                    alpha=0.8,
                    edgecolors='black',
                    linewidth=0.8)

# 扩展坐标轴范围（增加30%的边距）
x_range = x.max() - x.min()
y_range = y.max() - y.min()
z_range = z.max() - z.min()

x_margin = x_range * 0.3
y_margin = y_range * 0.3
z_margin = z_range * 0.3

ax.set_xlim(x.min() - x_margin, x.max() + x_margin)
ax.set_ylim(y.min() - y_margin, y.max() + y_margin)
ax.set_zlim(z.min() - z_margin, z.max() + z_margin)

# 更智能的文本偏移函数
def get_smart_text_offset(i, x_val, y_val, z_val, all_points, base_offset=0.04):
    """根据点的位置智能选择偏移方向"""
    # 计算点的大致象限
    x_center = np.mean(all_points[:, 0])
    y_center = np.mean(all_points[:, 1])
    z_center = np.mean(all_points[:, 2])
    
    # 根据点相对于中心的位置决定偏移方向
    x_dir = 1 if x_val < x_center else -0.1
    y_dir = 1 if y_val < y_center else -0.1
    z_dir = 1 if z_val < z_center else -0.1
    
    # 添加一些随机性避免完全对称
    random_factor = 0.1
    offset_x = x_dir * base_offset * (1 + random_factor * (i % 3))
    offset_y = y_dir * base_offset * (1 + random_factor * ((i + 1) % 3))
    offset_z = z_dir * base_offset * (1 + random_factor * ((i + 2) % 3))
    
    return offset_x, offset_y, offset_z

# 添加文本标注 - 使用智能偏移
for i, celltype in enumerate(celltypes):
    offset_x, offset_y, offset_z = get_smart_text_offset(i, x[i], y[i], z[i], points, base_offset=0.06)
    
    # 根据文本长度调整字体大小
    text_size = 6 if len(celltype) > 15 else 7
    
    ax.text(x[i] + offset_x, y[i] + offset_y, z[i] + offset_z, 
            celltype, 
            fontsize=text_size, 
            ha='left' if offset_x > 0 else 'right',
            va='bottom' if offset_y > 0 else 'top',
            bbox=dict(boxstyle='round,pad=0.15', 
                     facecolor='lightyellow', 
                     alpha=0.85, 
                     edgecolor='gray',
                     linewidth=0.3))

# 设置标签和标题
ax.set_xlabel('AP', fontsize=13, fontweight='bold', labelpad=18)
ax.set_ylabel('Tip', fontsize=13, fontweight='bold', labelpad=18)
ax.set_zlabel('Stalk', fontsize=13, fontweight='bold', labelpad=18)
ax.set_title('3D view of Celltypes', 
             fontsize=16, fontweight='bold', pad=25)

# 添加网格
ax.grid(True, alpha=0.3)

# 设置多个视角并保存
angles = [(25, 45), (15, 135), (30, 225)]  # 不同的视角

for j, (elev, azim) in enumerate(angles):
    ax.view_init(elev=elev, azim=azim)
    plt.tight_layout()
    # plt.savefig(f'3d_scatter_celltypes_view_{j+1}.pdf', 
    #             dpi=300, 
    #             bbox_inches='tight', 
    #             format='pdf')
    print(f"已保存视角 {j+1}: elev={elev}, azim={azim}")

# 显示最后一个视角
fig_dir='external_inputs/OnlyEndo'
plt.savefig(fig_dir+"/Endo_tip_ap_stalk_signature_scatter3D.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()



In [ ]:
####### 展示下 Celltype, Celltype_general, Celltype_sub 的 UMAP图
sc.pl.umap(adata, color=['tissue','leiden','Celltype_general', 'Celltype_sub', 'Celltype'], show=True, ncols =2, wspace=0.3 , hspace=0.2)  # wspace=0.3,legend_loc ='on data'
fig_dir='external_inputs/OnlyEndo'
plt.savefig(fig_dir+"/Endo_Celltype_umap.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()


In [ ]:
####### 根据新的去掉T、B那几个cluster后的adata 重新生成新的 cluster_to_celltype 和 cluster_to_celltype_general 的 dict
cluster_to_celltype = adata.obs.groupby('leiden')['Celltype'].first().to_dict()
cluster_to_celltype = {str(k): v for k, v in cluster_to_celltype.items()}
cluster_to_celltype_general = adata.obs.groupby('leiden')['Celltype_general'].first().to_dict()
cluster_to_celltype_general = {str(k): v for k, v in cluster_to_celltype_general.items()}  # 这样能得到新的 dict



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

Nat_color_list = ['#E64B35',  # Nature红
        '#4DBBD5',  # Nature蓝
        '#00A087',  # Nature绿
        '#3C5488',  # Nature深蓝
        '#F39B7F',  # 珊瑚色
        '#8491B4',  # 淡紫色
        '#91D1C2',  # 薄荷绿
        '#DC0000',  # 正红
        '#7E6148',  # 棕色
        '#B09C85',  # 米色
        '#F4649D',  # 粉红
        '#7C6A9C',  # 紫罗兰
        '#D9A428',  # 金黄色
        '#59A14F',  # 翠绿
        '#AF7AC4',  # 浅紫
        '#FFBB78',  # 杏色
        '#98DF8A',  # 嫩绿
        '#F7B6D2',  # 浅粉
        '#C5B0D5',  # 淡紫
        '#FF9896',  # 浅红
        '#9467BD',  # 深紫
        '#8C564B',  # 赭石
        '#E377C2',  # 玫红
        '#7F7F7F',  # 中灰
        '#BCBD22',  # 黄绿
        '#17BECF',  # 青绿
        '#AEC7E8',  # 天蓝
        '#C49C94',  # 肉色
        '#DBDB8D',  # 淡黄
        '#9EDAE5',  # 浅青
        '#F5B3A5',  # 浅珊瑚
        '#B3B3B3',  # 浅灰
        '#C2B0B0',  # 灰粉
        '#8C8C8C',  # 深灰
        '#6B8E23',  # 橄榄绿
        '#FF8C00',  # 暗橙
        '#4682B4'   # 钢蓝
]


def expand_color_list(colors, target_len):
    """将颜色列表通过插值扩充到目标长度"""
    if target_len <= len(colors):
        return colors[:target_len]
    
    # 将hex转RGB
    def hex_to_rgb(hex_color):
        hex_color = hex_color.lstrip('#')
        return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
        
    # RGB转hex
    def rgb_to_hex(rgb):
        return '#{:02X}{:02X}{:02X}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))
    
    rgb_colors = [hex_to_rgb(c) for c in colors]
    rgb_colors = np.array(rgb_colors)
    
    # 在颜色环上均匀插值
    indices = np.linspace(0, len(rgb_colors) - 1, target_len)
    new_colors = []
    
    for i in indices:
        idx_low = int(np.floor(i))
        idx_high = int(np.ceil(i))
        if idx_low == idx_high:
            new_colors.append(rgb_colors[idx_low])
        else:
            # 线性插值
            frac = i - idx_low
            interp_color = (1 - frac) * rgb_colors[idx_low] + frac * rgb_colors[idx_high]
            new_colors.append(interp_color)
    
    # 添加轻微扰动避免颜色过于相似（可选，增强区分度）
    np.random.seed(42)  # 固定随机种子保证可重复性
    for i in range(len(new_colors)):
        # 轻微调整亮度/饱和度，幅度<5%
        noise = np.random.uniform(-0.05, 0.05, 3)
        new_colors[i] = np.clip(new_colors[i] + new_colors[i] * noise, 0, 255)
    
    return [rgb_to_hex(c) for c in new_colors]

# 扩充到需要的不同的颜色个数
Nat_color_list = expand_color_list(Nat_color_list, 46)



In [ ]:
#### 画UMAP之前先抽样
adata_plot = sc.pp.subsample(
    adata,
    n_obs=300000,
    copy=True,
    random_state=0
)


In [ ]:
######### 画 Endothelial Atlas 的 Celltype UMAP图，现在先不加那个圈圈，PDF也先不存
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
# 1. 基础检查
required_cols = ['leiden', 'Celltype_general', 'Celltype']
for col in required_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"adata.obs 中缺少必要列: {col}")

# 确保 leiden 是字符串（Scanpy 推荐）
adata.obs['leiden'] = adata.obs['leiden'].astype(str)
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)
# 根据adata.obs['leiden'] 产生 leiden color list, 参考Nature系列的配色
# ================================
# 2. 构建 Leiden → 颜色映射
# ================================
leiden_categories = sorted(
    adata.obs['leiden'].unique(),
    key=lambda x: int(x)
)
if len(leiden_categories) != len(Nat_color_list):
    raise ValueError("Nat_color_list 数量必须等于 Leiden cluster 数量")

leiden_color_dict = dict(zip(leiden_categories, Nat_color_list))
# 告诉 scanpy 使用这套颜色
adata.uns['leiden_colors'] = [
    leiden_color_dict[c] for c in leiden_categories
]
# ================================
# 3. 构建 Celltype_C 列
# ================================

adata.obs['Celltype_C'] = (
    adata.obs['Celltype'] +
    "_C" +
    adata.obs['leiden']
)
# ================================
# 4. 构建 Celltype_C → 颜色映射（来自 Leiden）
# ================================
celltypeC_color_dict = {}
for leiden_id in leiden_categories:
    mask = adata.obs['leiden'] == leiden_id
    celltypeC_values = adata.obs.loc[mask, 'Celltype_C'].unique()
    for val in celltypeC_values:
        celltypeC_color_dict[val] = leiden_color_dict[leiden_id]

# ================================
# 5. 开始绘图
# ================================
plt.rcParams['figure.figsize'] = (13, 6.2)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='leiden',
    palette=Nat_color_list,
    ax=ax,
    show=False,
    legend_loc='on data',   # 不用scanpy默认legend
    s=10,
    alpha=0.8
)
ax.set_title('Endothelial Atlas', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
# ----------------------
# 5. 添加带箭头的轴标签
# ----------------------
# 计算箭头长度（基于UMAP坐标范围，确保箭头比例协调）
x_range = ax.get_xlim()
y_range = ax.get_ylim()
arrow_length_x = (x_range[1] - x_range[0]) * 0.1  # 占X轴范围10%
arrow_length_y = (y_range[1] - y_range[0]) * 0.1  # 占Y轴范围10%
# 添加UMAP_1箭头（x轴0点向右）
ax.annotate(
    'UMAP_1',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(arrow_length_x, 0),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='left',  # 文本对齐方式
    va='center'
)
# 添加UMAP_2箭头（y轴0点向上，竖直文本）
ax.annotate(
    'UMAP_2',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(0, arrow_length_y),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='center',  # 文本对齐方式
    va='bottom',
    rotation=90   # 文本竖直排布
)

# ================================
# 8. 构建分组 legend（按 Celltype_general）
# ================================
general_groups =['EndoMT', 'CapEC', 'VenEC', 'MCs', 'ArtEC', 'LEC']  # ['Peri', 'VenEC', 'LEC', 'EndoMT-II', 'EndoMT-I','ArtEC','CapEC','SMC']  ## LEC 就是 LymEC
col1_generals = [ 'VenEC', 'CapEC','LEC' ]
col2_generals = ['EndoMT','ArtEC','MCs']
def build_legend_handles(generals):
    handles = []
    labels = []
    for g in generals:
        subset = adata.obs[adata.obs['Celltype_general'] == g]
        unique_celltypeC = sorted(subset['Celltype_C'].unique())
        # 添加 general 标题（黑色）
        handles.append(mpatches.Patch(color='white', label=g))
        labels.append(g)
        # 添加每个 Celltype_C
        for ct in unique_celltypeC:
            handles.append(
                mpatches.Patch(
                    color=celltypeC_color_dict[ct],
                    label=ct
                )
            )
            labels.append("  " + ct)  
    return handles, labels


# 左 legend
handles1, labels1 = build_legend_handles(col1_generals)
legend1 = ax.legend(
    handles1,
    labels1,
    bbox_to_anchor=(1.01, 1.01),
    loc='upper left',
    frameon=False,
    fontsize=9
)
ax.add_artist(legend1)
# 右 legend
handles2, labels2 = build_legend_handles(col2_generals)
legend2 = ax.legend(
    handles2,
    labels2,
    bbox_to_anchor=(1.32, 1.01),
    loc='upper left',
    frameon=False,
    fontsize=9
)
# ----------------------
# 8. 计算每个 Celltype_general 对应的UMAP坐标边界（用于画圈）
# ----------------------
def get_cluster_bounds(adata, group_col):
    """计算每个分组的UMAP坐标边界（中心+宽高）"""
    bounds = {}
    for group in adata.obs[group_col].unique():
        # 获取该分组的UMAP坐标
        umap_coords = adata.obsm['X_umap'][adata.obs[group_col] == group]
        if len(umap_coords) == 0:
            continue
        # 计算中心坐标
        center_x = np.mean(umap_coords[:, 0])
        center_y = np.mean(umap_coords[:, 1])
        # 计算边界（用于确定圆圈大小）
        min_x, max_x = np.min(umap_coords[:, 0]), np.max(umap_coords[:, 0])
        min_y, max_y = np.min(umap_coords[:, 1]), np.max(umap_coords[:, 1])
        # 计算宽度和高度（扩大1.2倍，让圈更宽松）
        width = (max_x - min_x) * 1.2
        height = (max_y - min_y) * 1.2
        bounds[group] = {
            'center': (center_x, center_y),
            'width': width,
            'height': height 
        }
    return bounds

###################### 这里计算边界，然后画圈的暂时先不要
# 计算每个Celltype_general的UMAP边界
cluster_bounds = get_cluster_bounds(adata, 'Celltype_general')

# ----------------------
# 9. 为每个Celltype_general添加弯曲虚线圆圈和文本标签
general_colors = {
    # 'Peri': '#1f77b4',       # 深海蓝（Cell经典主色）
    'VenEC': '#ff7f0e',    # 暖橙（Nature常用强调色）
    'EndoMT': '#2ca02c',# 森林绿（Science功能色）
    'CapEC': '#d62728',    # 猩红（顶级期刊高亮色）
    'LEC': '#9467bd',    # 紫罗兰（Cell子刊常用）
    'MCs': '#8c564b',      # 棕褐（Nature子刊纹理色）
    'ArtEC': '#e377c2'    # 蔷薇粉（Science子刊对比色）
   #  'EndoMT-I': '#7f7f7f'  # 石墨灰（期刊中性对比色）
}

# ----------------------
for general_type, color in general_colors.items():
    if general_type not in cluster_bounds:
        continue
        
    # 获取该分组的边界信息
    bounds = cluster_bounds[general_type]
    center_x, center_y = bounds['center']
    width = bounds['width']
    height = bounds['height']
    # 创建弯曲的椭圆虚线框（模拟弯曲圈）
    ellipse = mpatches.Ellipse(
        (center_x, center_y),
        width=width,
        height=height,
        edgecolor=color,
        facecolor='none',
        linestyle='--',  # 虚线
        linewidth=2,     # 线宽
        alpha=0.8        # 透明度
    )
    ax.add_patch(ellipse)
    # 添加文本标签（在圈的右侧/上方，避免遮挡）
    # 计算标签位置（圈的右侧偏移一点）
    label_x = center_x + width/2 + (x_range[1]-x_range[0])*0.02
    label_y = center_y
    # 添加文本
    ax.text(
        label_x, label_y,
        general_type,
        fontsize=12,
        fontweight='bold',
        color=color,
        ha='left',
        va='center',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor=color, alpha=0.7)
    )

# 10. 保存
plt.tight_layout()
fig_dir='external_inputs/OnlyEndo'
plt.savefig(
    fig_dir+"Endothelial_umap_leiden_Celltype.pdf",
    dpi=300,
    bbox_inches='tight',
    format='pdf'
)
#plt.savefig(fig_dir+"/Endothelial_umap_leiden_Celltype.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()



In [ ]:
########## 接下来画那 8个 signature score 的柱状图
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
# 设置Nature/Cell期刊风格的参数
plt.style.use('default')
rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})
# 1. 提取数据框并添加Celltype信息
adata_meta = adata.obs[['leiden','Celltype_general','Celltype','MCs_1', 'LEC_1', 'VenEC_1', 'CapEC_1', 'ArtEC_1', 'EndoMT_1']].copy()
# rename_dict = {
#     'score_Tip': 'Tip',
#     'score_antigen_pre': 'AP',
#     'score_stalk': 'Stalk'
# }
# adata_meta = adata_meta.rename(columns=rename_dict) 
adata_meta['Celltype'] = adata_meta['Celltype'].astype(str) + '_C' + adata_meta['leiden'].astype(str)
# 定义leiden到Celltype的映射字典

# 确保leiden列是字符串类型，方便处理
adata_meta['leiden'] = adata_meta['leiden'].astype(str)
# 3. 按照leiden分组计算平均值
# 注意：这里修正了列名大小写问题
feature_columns = ['MCs_1', 'LEC_1', 'VenEC_1', 'CapEC_1', 'ArtEC_1', 'EndoMT_1']      ##  ['Tip', 'AP', 'Stalk'], 可以替换
# 检查列名是否存在，确保大小写匹配
available_columns = [col for col in feature_columns if col in adata_meta.columns]
grouped_mean = adata_meta.groupby(['Celltype'])[available_columns].mean().reset_index()
grouped_mean['leiden'] = grouped_mean['Celltype'].str.split('_C').str[-1].astype(int)
grouped_mean = grouped_mean.sort_values(['Celltype','leiden'])
print(grouped_mean[['leiden', 'Celltype']].to_string(index=False))
# 5. 可视化设置
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}") 
# 设置子图布局：4行10列
n_rows = 6
n_cols = 8
n_subplots = n_rows * n_cols
# 如果leiden数量少于子图数量，调整布局
if n_leiden < n_subplots:
    n_cols = int(np.ceil(n_leiden / n_rows))
    n_subplots = n_rows * n_cols

colors =  plt.cm.Set2(np.linspace(0, 1, len(available_columns)))
# 6. 创建图形
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2*n_cols, 1.2*n_rows))
axes = axes.flatten()
# 设置图形标题
fig.suptitle('Signature score of subEndo markers',  # Signature score of subFibro(long list)
             fontsize=12, fontweight='bold', y=1.02)

# 7. 绘制每个leiden的柱状图（添加数值标签）
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):
        break
    
    leiden = row.leiden
    celltype = row.Celltype
    ax = axes[i]
    
    # 提取当前leiden的数据
    leiden_data = grouped_mean.loc[grouped_mean['leiden'] == leiden, available_columns].iloc[0]
    
    # 创建柱状图
    x_pos = np.arange(len(available_columns))
    bars = ax.bar(x_pos, leiden_data.values, color=colors, edgecolor='black', linewidth=0.5)
    
    # 添加数值标签到每个柱子顶部
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        # 确定标签位置
        height = bar.get_height()
        
        # 如果值太小，将标签放在柱子内部顶部
        if height < grouped_mean[available_columns].values.max() * 0.05:
            # 小值：标签放在柱子内部，白色字体
            text_y = height * 0.8  # 放在柱子高度的80%位置
            color = 'black' # 小的值也还是 black
            va = 'bottom'
        else:
            # 正常值：标签放在柱子顶部上方
            text_y = height + grouped_mean[available_columns].values.max() * 0.01  # 稍微高于柱子
            color = 'black'
            va = 'bottom'
        
        # 格式化数值显示（根据数值大小选择格式）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加文本标签
        ax.text(
            bar_idx, text_y, text,
            ha='center', va=va,
            fontsize=6,  # 使用较小字体
            color=color,
            fontweight='bold' if color == 'white' else 'normal',
            rotation=90
        )
    
    # 设置子图标题（包含leiden和Celltype信息）
    # 缩短Celltype显示，只显示主要部分
    if len(celltype) > 20:
        # 如果Celltype太长，只显示前部分
        short_celltype = celltype # celltype[:17] + "..."
    else:
        short_celltype = celltype
    
    title_text =f'{short_celltype}'  # f'L{leiden}\n{short_celltype}'
    ax.set_title(title_text, fontsize=6, fontweight='bold', pad=3)
    
    # 隐藏x轴刻度
    ax.set_xticks([])
    
    # 设置y轴标签
    if i % n_cols == 0:  # 每行最左边的图显示y轴标签
        ax.set_ylabel('Sig Score', fontsize=8)
    
    # 设置y轴范围，为顶部标签留出空间
    y_max = grouped_mean[available_columns].values.max() * 1.15  # 增加15%的空间用于标签
    ax.set_ylim([0, y_max])
    
    # 添加网格线（仅y方向，更简洁）
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # 移除顶部和右侧边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 根据Celltype前缀添加背景色区分
    celltype_prefix =  celltype.split('_')[0]     # get_celltype_prefix(celltype) , 这个函数不在了
    if celltype_prefix == 'ArtEC':  # iFibro
        ax.set_facecolor('#FFF5E6')  # 浅橙色背景
    elif celltype_prefix == 'CapEC':  # eFibro
        ax.set_facecolor('#E6F7FF')  # 浅蓝色背景
    elif celltype_prefix == 'EndoMT':  # Myfibro
        ax.set_facecolor('#F0E6FF')  # 浅紫色背景
    # elif celltype_prefix == 'EndoMT-I':  # qFibro
    #     ax.set_facecolor('#E6FFE6')  # 浅绿色背景
    elif celltype_prefix == 'LEC':  # eFibro
        ax.set_facecolor("#E58606")  
    # elif celltype_prefix == 'Peri':  # Myfibro
    #     ax.set_facecolor("#BEBADA")  
    elif celltype_prefix == 'MCs':  # qFibro
        ax.set_facecolor("#9C9C5E")  
    elif celltype_prefix == 'VenEC':  # qFibro
        ax.set_facecolor("#F3D4F4")  # 

# 8. 隐藏多余的子图
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# 9. 创建统一的图例
# 使用简化版特征名称作为图例标签
# feature_short_names = []
# for col in available_columns:
#     if 'Tip' in col:
#         feature_short_names.append('Tip')
#         # feature_short_names.append('Myfibro_l')
#     elif 'AP' in col:
#         feature_short_names.append('AP')
#     elif 'Stalk' in col:
#         feature_short_names.append('Stalk')
feature_short_names = available_columns  # ['SMC', 'Peri', 'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I', 'EndoMT-II']


# 创建自定义的图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, color=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(available_columns))]


# 创建Celltype分组图例
celltype_prefix_colors = {
    1: ('ArtEC', '#FFF5E6'),
    2: ('CapEC', '#E6F7FF'),
    3: ('EndoMT', '#F0E6FF'),
   #  4: ('EndoMT-I', '#E6FFE6'),
    4: ('LEC', '#E58606'),
   # 6: ('Peri', '#BEBADA'),
    5: ('MCs', '#9C9C5E'),
    6: ('VenEC', '#F3D4F4')
}

# 创建Celltype图例句柄
celltype_handles = []
celltype_labels = []

for prefix in sorted(celltype_prefix_colors.keys()):
    label, color = celltype_prefix_colors[prefix]
    handle = plt.Rectangle((0,0),1,1, color=color, edgecolor='gray', linewidth=0.5, alpha=0.7)
    celltype_handles.append(handle)
    celltype_labels.append(label)

########## 都是同一个图例，就用同一个图例就好，放在中间
# Subtypes图例（左下区域）
fig.legend(legend_handles, feature_short_names, 
           loc='lower center', 
           bbox_to_anchor=(0.25, 0.01),  # 底部左侧
           ncol=len(available_columns),  # 水平排列
           fontsize=8,
           title='marker_list',
           title_fontsize=9,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(celltype_handles, celltype_labels,
           loc='lower center',
           bbox_to_anchor=(0.7, 0.01),  # 底部右侧
           ncol=8,  # 水平排列
           title='Celltype',
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.11,left=0.05, right=0.9, hspace=0.4, wspace=0.3)

# 12. 保存图形
output_path = 'external_inputs/Endo_leiden_subtype_signature_scores_L.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"图形已保存到: {output_path}")
# 13. 显示图形
plt.show()



In [ ]:
####### 产画 marker dotplot 需要的文件， 函数在别的脚本里

genes_to_check = Endo_marker_all
meta_data = prepare_marker_data(
    adata, 
    genes_to_check, 
    celltype_col='Celltype', 
    output_path='external_inputs/celltype_gene_exp.csv'
)



In [ ]:
# genes_to_check
os.getcwd()


In [ ]:
# 将 age 转成 float，用来记录哪些有 Age信息
def isstr(x):
    try: 
        float(x)
        return False
    except ValueError as e:
        return True


adata.obs["age"] = adata.obs["age"].astype(str)
def assign_age_group(age):
    # age = str(age).lower()
    if 'Young adult' in age:
        return 'B'
    elif 'nfant' in age:
        return 'A'
    elif 'E' in age:
        return 'A'
    elif age == 'Aged':
        return 'H'
    elif 'month' in age:
        return 'A'
    elif 'Embr' in age:
        return 'A'
    try:
        age = float(age)  # 转换为浮动类型
        if 0<=age<20:
            return 'A'
        elif 20 <= age < 30:
            return 'B'
        elif 30 <= age < 40:
            return 'C'
        elif 40 <= age < 50:
            return 'D'
        elif 50 <= age < 60:
            return 'E'
        elif 60 <= age < 70:
            return 'F'
        elif 70 <= age < 90:
            return 'G'
        elif age >= 90:
            return 'H'
    except ValueError:
        return 'Unknown'  # 对age 信息重新进行分群 


adata.obs['Age_group']  = adata.obs['age'].apply(assign_age_group)
adata_age  = adata[~np.array(list(map(lambda x: isstr(x),adata.obs['age'])))]
adata_age.obs['age'] = adata_age.obs['age'].astype(float) 
adata_age.obs['Age'] = adata_age.obs['age']
adata_age.write('/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo_age.h5ad')



In [ ]:
######### 对integration 和 batch removal 效果的评估，使用 scib 包中的现成函数

# 初始化 Benchmarker
bm = Benchmarker(
    adata,
    batch_key='tissue',               # 你的批次信息列名
    label_key='leiden',              # 你的细胞类型/聚类结果列名
    embedding_obsm_keys=['X_umap'],   # 用于评估的降维结果，可以是 'X_pca' 或 'X_scVI',这里应该用 'X_umap' 等
    n_jobs=-1,                       # 使用所有 CPU 核心加速，若报错可改为 8
)

# 运行评估 (这可能需要几分钟时间，取决于数据量)
bm.benchmark()
results_df = bm.get_results(min_max_scale=False)       #   X_pca	0.85	0.72	0.95	0.42	0.78	1.0 ; 一个方法的 benchmark 结果大概长这样



In [ ]:
# sorted(dir()) 


In [ ]:
######### Divide the adata_age info  healthy samples 





In [ ]:
####### 注释好后画一下 Celltype_general, Celltype_sub, Celltype 的图，还有那个 Leiden值在UMAP上，legend是 Celltype的图
##### 画 Celltype_general 的代码 
plt.rcParams['figure.figsize'] = (10, 5)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='Celltype',
    # palette=Nat_color_list,
    ax=ax,
    show=False,
    s=10,
    alpha=0.8
)
ax.set_title('Celltype', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig_dir='external_inputs/OnlyEndo'
# plt.savefig(
#     fig_dir+"Endo_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Endo_Celltype.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()
